# LeetCode 272 — Closest Binary Search Tree Value II bằng Python (Hard, Premium)

> **Lưu ý:** đây là bài **Premium** — trên LeetCode bài này bị khóa, cần tài khoản trả phí mới xem đề và nộp bài được. Phát biểu đề, ví dụ, constraints và signature trong notebook bám theo bản LeetCode; chỗ nào là **giả định** của notebook (không ghi trong đề) sẽ được nói rõ.

Notebook này giảng nhập môn bài toán:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def closestKValues(self, root: Optional[TreeNode], target: float, k: int) -> List[int]:

```

Mục tiêu của bài học:

1. Hiểu **cây nhị phân** (binary tree) từ con số 0: node, root, con trái/phải, lá, chiều cao; tự dựng cây từ list kiểu LeetCode và **in cây ra màn hình** để nhìn thấy nó.
2. Hiểu **cây nhị phân tìm kiếm** (BST) và hai tính chất vàng: (a) duyệt **in-order** cho ra **dãy tăng dần**; (b) tìm một giá trị chỉ cần đi **một đường** từ root xuống, tốn `O(h)`.
3. Ôn bài **#270** (tìm **một** giá trị gần nhất) và hai khái niệm **predecessor / successor** của `target` — chìa khóa của lời giải chính.
4. Đi qua 4 họ lời giải theo thứ tự tự nhiên: **sắp xếp theo khoảng cách** `O(n log n)` → **max-heap kích thước k** `O(n log k)` → **khai thác dãy tăng dần**: bisect + hai con trỏ, hoặc cửa sổ trượt dừng sớm `O(n)` → **hai stack predecessor/successor** `O(h + k)` (cách chính, trả lời câu follow-up).
5. Học các công cụ dùng lại được ở rất nhiều bài khác: `sorted(key=...)`, `heapq`, `bisect`, `deque`, stack, **duyệt in-order bằng vòng lặp** (BST iterator), "hai con trỏ trên cây".
6. Biết tự kiểm thử bằng **brute force trên BST ngẫu nhiên** và xử lý đúng chuyện **hòa khoảng cách**.

## 1. Đề bài

Đề LeetCode (dịch sát nghĩa):

> Cho `root` của một **cây nhị phân tìm kiếm** (binary search tree — BST), một giá trị `target` (số thực) và một số nguyên `k`, hãy trả về **`k` giá trị trong BST gần `target` nhất**. Bạn có thể trả về kết quả theo **thứ tự bất kỳ**.
>
> Đề **đảm bảo** rằng chỉ có **duy nhất một tập** gồm `k` giá trị trong BST gần `target` nhất.

**Example 1**

```text
Input: root = [4,2,5,1,3], target = 3.714286, k = 2
Output: [4,3]
```

**Example 2**

```text
Input: root = [1], target = 0.000000, k = 1
Output: [1]
```

**Constraints**

- Số node trong cây là `n`.
- `1 <= k <= n <= 10^4`.
- `0 <= Node.val <= 10^9`
- `-10^9 <= target <= 10^9`

**Follow up:** Giả sử BST **cân bằng** (balanced). Bạn có giải được với thời gian chạy **nhỏ hơn `O(n)`** (`n` = tổng số node) không?

### Ký hiệu dùng suốt notebook

- `n` = số node của cây; `k` = số giá trị cần trả về (`1 <= k <= n`).
- `h` = **chiều cao** của cây = số node trên đường dài nhất từ root xuống một lá. Cây cân bằng thì `h ≈ log₂ n` (với `n = 10^4` thì `h = 14`), cây lệch thành một "sợi dây" thì `h = n`.
- **Khoảng cách** từ một giá trị `x` tới `target` là `|x - target|` (trị tuyệt đối của hiệu). "Gần nhất" = khoảng cách nhỏ nhất.
- **Giả định của notebook:** các giá trị trong BST **đôi một khác nhau** — đúng theo định nghĩa BST quen thuộc của LeetCode (trái **nhỏ hơn hẳn**, phải **lớn hơn hẳn**). Constraints của đề không ghi riêng dòng này nên ta nói rõ ở đây.

### 1.1 Đọc kỹ Example 1

Cây `[4,2,5,1,3]` trông như sau (cách đọc list sẽ học ở mục 2):

```text
        4
       / \
      2   5
     / \
    1   3
```

Tính khoảng cách của từng giá trị tới `target = 3.714286`:

| x | x − target | \|x − target\| | Xếp hạng (gần → xa) |
|---:|---:|---:|:---:|
| 4 | +0.285714 | 0.285714 | 1 |
| 3 | −0.714286 | 0.714286 | 2 |
| 5 | +1.285714 | 1.285714 | 3 |
| 2 | −1.714286 | 1.714286 | 4 |
| 1 | −2.714286 | 2.714286 | 5 |

Với `k = 2`, ta lấy hai giá trị xếp hạng 1 và 2 → `{4, 3}`. Vì đề cho phép **thứ tự bất kỳ**, trả `[4,3]` hay `[3,4]` đều đúng.

Để ý: `target` nằm **giữa** 3 và 4, và hai đáp án chính là hai giá trị "kẹp" lấy `target`. Đây không phải ngẫu nhiên — mục 5 và mục 8 sẽ biến quan sát này thành thuật toán.

### 1.2 "Chỉ có duy nhất một tập đáp án" nghĩa là gì?

Giả sử `target = 3.5` và `k = 1`. Khi đó `|3 − 3.5| = |4 − 3.5| = 0.5` → **hòa**: cả `{3}` lẫn `{4}` đều là "1 giá trị gần nhất". Đề **cam kết không bao giờ cho input như vậy**.

Nhưng `target = 3.5`, `k = 2` thì vẫn hợp lệ: đáp án duy nhất là `{3, 4}` (hòa xảy ra **bên trong** đáp án, không ảnh hưởng tập kết quả). Nói gọn: nếu sắp các khoảng cách tăng dần `d₁ ≤ d₂ ≤ ... ≤ dₙ` thì đề đảm bảo `d_k < d_(k+1)` — không hòa ở **ranh giới** giữa phần tử thứ `k` và thứ `k+1`.

Khi nào hai giá trị khác nhau `x < y` hòa khoảng cách? `|x − t| = |y − t|` ⇔ `t` nằm **chính giữa**: `t = (x + y) / 2`. Ta sẽ dùng nhận xét này để sinh test ngẫu nhiên không bị hòa ở mục 12.

In [1]:
# Khởi động: chưa cần cây, chỉ cần các giá trị của Example 1
values = [4, 2, 5, 1, 3]
target = 3.714286
k = 2

for x in values:
    print(f"x = {x}   x - target = {x - target:+.6f}   |x - target| = {abs(x - target):.6f}")

# Hòa khoảng cách: target nằm chính giữa hai giá trị 3 và 4
t = 3.5
print("\n|3 - 3.5| =", abs(3 - t), "  |4 - 3.5| =", abs(4 - t), "  -> hòa:", abs(3 - t) == abs(4 - t))
print("điểm chính giữa (3 + 4) / 2 =", (3 + 4) / 2)

x = 4   x - target = +0.285714   |x - target| = 0.285714
x = 2   x - target = -1.714286   |x - target| = 1.714286
x = 5   x - target = +1.285714   |x - target| = 1.285714
x = 1   x - target = -2.714286   |x - target| = 2.714286
x = 3   x - target = -0.714286   |x - target| = 0.714286

|3 - 3.5| = 0.5   |4 - 3.5| = 0.5   -> hòa: True
điểm chính giữa (3 + 4) / 2 = 3.5


## 2. Nền tảng: Cây nhị phân từ căn bản

### 2.1 Từ vựng

**Cây nhị phân** (binary tree) là cấu trúc gồm các **node** (nút). Mỗi node có:

- `val`: giá trị lưu ở node.
- `left`: con **trái** (một node khác) hoặc `None` nếu không có.
- `right`: con **phải** hoặc `None`.

Dùng cây Example 1 ở trên để gọi tên:

| Thuật ngữ | Nghĩa | Ví dụ trên cây Example 1 |
|---|---|---|
| **root** (gốc) | node trên cùng, không có cha | `4` |
| **child** (con) / **parent** (cha) | node ngay dưới / ngay trên | `2` là con trái của `4`; `4` là cha của `2` |
| **leaf** (lá) | node không có con nào | `1`, `3`, `5` |
| **subtree** (cây con) | một node cùng toàn bộ con cháu của nó | cây con gốc `2` gồm `{2, 1, 3}` |
| **height** (chiều cao) `h` | số node trên đường dài nhất từ root xuống lá | `4 → 2 → 1` có 3 node → `h = 3` |

Khác với Python `list` (truy cập bằng chỉ số `a[i]`), ở cây ta chỉ cầm **node root** và đi xuống bằng `.left` / `.right`. Muốn tới node `3` phải đi `root.left.right`.

LeetCode định nghĩa node bằng class `TreeNode` (cho sẵn, được comment ở đầu đề). Ta định nghĩa lại **y hệt** để chạy trong notebook.

`Optional[TreeNode]` trong signature (từ module `typing`) nghĩa là "hoặc là một `TreeNode`, hoặc là `None`"; `List[int]` nghĩa là "list các số nguyên". Đây chỉ là **chú thích kiểu** (type hint) cho người đọc, Python không bắt buộc.

Cell dưới cũng `import` sẵn mọi thứ notebook sẽ dùng (đều là thư viện chuẩn); từng công cụ sẽ được giảng khi gặp.

In [2]:
from typing import List, Optional
from collections import deque
from bisect import bisect_left
import heapq
import random
import sys


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


# Dựng tay cây Example 1:
#         4
#        / \
#       2   5
#      / \
#     1   3
root_ex1 = TreeNode(4, TreeNode(2, TreeNode(1), TreeNode(3)), TreeNode(5))

print(root_ex1.val)              # 4  (root)
print(root_ex1.left.val)         # 2  (con trái của 4)
print(root_ex1.left.right.val)   # 3  (đi: 4 -> trái -> phải)
print(root_ex1.right.left)       # None: node 5 không có con trái

4
2
3
None


### 2.2 LeetCode biểu diễn cây bằng list theo tầng (level-order)

Trên LeetCode, `root = [4,2,5,1,3]` được đọc **từng tầng, từ trái sang phải**:

```text
tầng 0:  4
tầng 1:  2, 5          (con trái, con phải của 4)
tầng 2:  1, 3          (con trái, con phải của 2; node 5 không còn phần tử nào -> không có con)
```

Chỗ trống được ghi là `null` (trong Python là `None`). Ví dụ `[1,null,2]` là: root `1`, **không có** con trái, con phải là `2`.

Quy tắc: lấy lần lượt từng node **thật** (không phải `null`) theo thứ tự tầng, mỗi node "tiêu thụ" 2 phần tử kế tiếp của list làm con trái và con phải. `null` không có con nên không tiêu thụ thêm phần tử nào. Các `null` ở cuối list thường được lược bỏ.

Để làm đúng thứ tự "node nào đến trước thì được gán con trước", ta dùng **hàng đợi** (queue): vào trước, ra trước (FIFO — First In, First Out). Python có sẵn `collections.deque` (đọc là "đéc" — double-ended queue, hàng đợi hai đầu):

- `q.append(x)`: thêm `x` vào **cuối** hàng.
- `q.popleft()`: lấy ra phần tử ở **đầu** hàng — nhanh `O(1)`, trong khi `list.pop(0)` phải dịch cả list nên chậm `O(n)`.
- `q[0]` xem phần tử đầu, `q[-1]` xem phần tử cuối (không lấy ra).

`deque` sẽ còn quay lại ở cách 3 (cửa sổ trượt) và Bài tập 5.

In [3]:
q = deque()
q.append("A")
q.append("B")
q.append("C")
print("hàng đợi:", list(q), "| đầu:", q[0], "| cuối:", q[-1])
print("ra trước:", q.popleft())   # A vào trước nên ra trước
print("ra tiếp :", q.popleft())   # B
print("còn lại :", list(q))       # ['C']
q.appendleft("Z")                 # deque thêm được cả ở ĐẦU hàng
print("thêm Z vào đầu:", list(q))

hàng đợi: ['A', 'B', 'C'] | đầu: A | cuối: C
ra trước: A
ra tiếp : B
còn lại : ['C']
thêm Z vào đầu: ['Z', 'C']


### 2.3 Helper: `build_tree` (list → cây) và `tree_to_list` (cây → list)

- `build_tree(values)`: tạo root từ `values[0]`, đưa vào queue; mỗi lần `popleft` một node thì đọc 2 phần tử tiếp theo làm con trái/phải (bỏ qua `None`); con thật được đưa vào cuối queue để lát nữa tới lượt nó nhận con.
- `tree_to_list(root)`: làm ngược lại để kiểm tra — duyệt theo tầng, ghi `None` cho chỗ trống, rồi cắt các `None` thừa ở cuối.

Nếu `build_tree` rồi `tree_to_list` ra lại đúng list ban đầu, ta tin được helper.

Ngoài cây Example 1, notebook dùng xuyên suốt một BST lớn hơn, gọi là **cây T** (10 node):

```text
T = [8,3,13,1,6,10,15,null,null,4,7,null,12]
```

Cả hai helper đều viết bằng **vòng lặp** (không đệ quy) nên dựng được cả cây sâu `10^4` tầng mà không lỗi.

In [4]:
def build_tree(values):
    # Dựng cây từ list level-order kiểu LeetCode (None = không có node).
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:      # con trái
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:      # con phải
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root


def tree_to_list(root):
    # Cây -> list level-order (để kiểm tra build_tree).
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            out.append(None)
            continue
        out.append(node.val)
        queue.append(node.left)
        queue.append(node.right)
    while out and out[-1] is None:      # bỏ None thừa ở cuối
        out.pop()
    return out


T_LIST = [8, 3, 13, 1, 6, 10, 15, None, None, 4, 7, None, 12]
root_T = build_tree(T_LIST)

print(tree_to_list(root_ex1))            # cây dựng tay ở 2.1 -> [4, 2, 5, 1, 3]
print(tree_to_list(root_T))
print(root_T.left.right.left.val)        # đi: 8 -> trái (3) -> phải (6) -> trái (4)
assert tree_to_list(build_tree([4, 2, 5, 1, 3])) == [4, 2, 5, 1, 3]
assert tree_to_list(root_T) == T_LIST
assert tree_to_list(build_tree([1, None, 2, None, 3])) == [1, None, 2, None, 3]
print("build_tree / tree_to_list OK")

[4, 2, 5, 1, 3]
[8, 3, 13, 1, 6, 10, 15, None, None, 4, 7, None, 12]
4
build_tree / tree_to_list OK


### 2.4 Helper: in cây dạng ASCII để **nhìn thấy** cây

Đọc list `[8,3,13,1,6,10,15,null,null,4,7,null,12]` rất khó hình dung. Hàm `print_tree` vẽ cây từ trên xuống:

- `/` nối một node với **con trái**, `\` nối với **con phải**;
- dấu `_` chỉ là "dây nối" kéo dài cho đẹp, không mang ý nghĩa.

Hàm `tree_lines` tự gọi lại chính nó cho từng con (đệ quy — học ở mục 3): vẽ cây con trái, vẽ cây con phải, rồi ghép hai hình cạnh nhau và đặt node hiện tại lên trên. **Không cần hiểu chi tiết** từng dòng của hàm này; chỉ cần dùng nó để nhìn cây.

In [5]:
def tree_lines(node):
    # Trả về (các dòng, bề rộng, chiều cao, vị trí cột của node gốc) của hình vẽ cây con.
    s = str(node.val)
    u = len(s)
    if node.left is None and node.right is None:
        return [s], u, 1, u // 2
    if node.right is None:                                   # chỉ có con trái
        lines, n, p, x = tree_lines(node.left)
        first = (x + 1) * " " + (n - x - 1) * "_" + s
        second = x * " " + "/" + (n - x - 1 + u) * " "
        shifted = [line + u * " " for line in lines]
        return [first, second] + shifted, n + u, p + 2, n + u // 2
    if node.left is None:                                    # chỉ có con phải
        lines, n, p, x = tree_lines(node.right)
        first = s + x * "_" + (n - x) * " "
        second = (u + x) * " " + "\\" + (n - x - 1) * " "
        shifted = [u * " " + line for line in lines]
        return [first, second] + shifted, n + u, p + 2, u // 2
    left, n, p, x = tree_lines(node.left)                    # có cả hai con
    right, m, q, y = tree_lines(node.right)
    first = (x + 1) * " " + (n - x - 1) * "_" + s + y * "_" + (m - y) * " "
    second = x * " " + "/" + (n - x - 1 + u + y) * " " + "\\" + (m - y - 1) * " "
    if p < q:
        left += [n * " "] * (q - p)
    elif q < p:
        right += [m * " "] * (p - q)
    lines = [first, second] + [a + u * " " + b for a, b in zip(left, right)]
    return lines, n + m + u, max(p, q) + 2, n + u // 2


def print_tree(root):
    if root is None:
        print("(cây rỗng)")
        return
    for line in tree_lines(root)[0]:
        print(line.rstrip())


print("Example 1:")
print_tree(root_ex1)
print("\nCây T:")
print_tree(root_T)
print("\nCây lệch [1,null,2,null,3]:")
print_tree(build_tree([1, None, 2, None, 3]))

Example 1:
  _4
 /  \
 2  5
/ \
1 3

Cây T:
  ___8_____
 /         \
 3_     __13_
/  \   /     \
1  6  10_   15
  / \    \
  4 7   12

Cây lệch [1,null,2,null,3]:
1
 \
 2
  \
  3


## 3. Nền tảng: Đệ quy và ba kiểu duyệt cây

### 3.1 Đệ quy trên cây

**Đệ quy** (recursion) là khi một hàm **gọi lại chính nó** trên một bài toán **nhỏ hơn**. Cây rất hợp với đệ quy vì: *cây = root + cây con trái + cây con phải*, mà hai cây con cũng là cây.

Một hàm đệ quy luôn có 2 phần:

1. **Base case** (trường hợp cơ sở): bài toán nhỏ tới mức trả lời ngay. Với cây, gần như luôn là `if node is None: return ...`.
2. **Recursive case**: gọi hàm cho `node.left`, `node.right`, rồi **kết hợp** hai kết quả với node hiện tại.

Ví dụ đếm số node và tính chiều cao:

```text
count(None)   = 0
count(node)   = 1 + count(node.left) + count(node.right)

height(None)  = 0
height(node)  = 1 + max(height(node.left), height(node.right))
```

Với Example 1: `height(1) = height(3) = height(5) = 1`, `height(2) = 1 + max(1, 1) = 2`, `height(4) = 1 + max(2, 1) = 3`.

### 3.2 Ba kiểu duyệt DFS

**DFS** (Depth-First Search — duyệt theo chiều sâu): đi xuống sâu hết một nhánh rồi mới quay lại nhánh khác. Cùng một khung đệ quy, chỉ cần đổi **vị trí của dòng "ghi node"** là ra ba thứ tự:

| Tên | Thứ tự | Example 1 cho ra |
|---|---|---|
| **pre-order** (tiền thứ tự) | node → trái → phải | `[4, 2, 1, 3, 5]` |
| **in-order** (trung thứ tự) | trái → node → phải | `[1, 2, 3, 4, 5]` ← **tăng dần!** |
| **post-order** (hậu thứ tự) | trái → phải → node | `[1, 3, 2, 5, 4]` |

In-order của Example 1 ra đúng dãy tăng dần. Đây chính là **tính chất quan trọng nhất của bài**, mục 4 sẽ giải thích vì sao.

In [6]:
def count_nodes(node):
    if node is None:                     # base case
        return 0
    return 1 + count_nodes(node.left) + count_nodes(node.right)


def height(node):
    if node is None:                     # base case: cây rỗng cao 0
        return 0
    return 1 + max(height(node.left), height(node.right))


def preorder(node, out):
    if node is None:
        return
    out.append(node.val)                 # ghi node TRƯỚC
    preorder(node.left, out)
    preorder(node.right, out)


def inorder(node, out):
    if node is None:
        return
    inorder(node.left, out)
    out.append(node.val)                 # ghi node Ở GIỮA
    inorder(node.right, out)


def postorder(node, out):
    if node is None:
        return
    postorder(node.left, out)
    postorder(node.right, out)
    out.append(node.val)                 # ghi node SAU CÙNG


for name, tree in [("Example 1", root_ex1), ("Cây T    ", root_T)]:
    print(f"{name}: n = {count_nodes(tree)}, h = {height(tree)}")
    for order_name, fn in [("pre-order ", preorder), ("in-order  ", inorder), ("post-order", postorder)]:
        out = []
        fn(tree, out)
        print("   ", order_name, out)

Example 1: n = 5, h = 3
    pre-order  [4, 2, 1, 3, 5]
    in-order   [1, 2, 3, 4, 5]
    post-order [1, 3, 2, 5, 4]
Cây T    : n = 10, h = 4
    pre-order  [8, 3, 1, 6, 4, 7, 13, 10, 12, 15]
    in-order   [1, 3, 4, 6, 7, 8, 10, 12, 13, 15]
    post-order [1, 4, 7, 6, 3, 12, 10, 15, 13, 8]


## 4. Nền tảng: Cây nhị phân tìm kiếm (BST)

### 4.1 Định nghĩa

Một cây nhị phân là **BST** (Binary Search Tree) nếu **với mọi node**:

- **mọi** giá trị trong cây con **trái** đều **nhỏ hơn** `node.val`;
- **mọi** giá trị trong cây con **phải** đều **lớn hơn** `node.val`.

Chú ý chữ **mọi**: không chỉ con trực tiếp mà cả cháu, chắt... Cây dưới đây **trông** ổn nếu chỉ nhìn từng cặp cha–con (`3 < 5`, `8 > 5`, `1 < 3`, `6 > 3`), nhưng **không phải BST**, vì `6` nằm trong cây con **trái** của root `5` mà lại lớn hơn `5`:

```text
        5
       / \
      3   8
     / \
    1   6      <- 6 > 5 nhưng nằm bên trái 5: vi phạm
```

### 4.2 Vì sao in-order của BST luôn tăng dần?

In-order ở một node làm 3 việc theo thứ tự: (1) liệt kê **cả cây con trái**, (2) ghi **node**, (3) liệt kê **cả cây con phải**.

Trong BST: mọi thứ ở (1) **nhỏ hơn** node, mọi thứ ở (3) **lớn hơn** node. Nên node đứng đúng chỗ giữa hai nhóm. Áp dụng **cùng lập luận** cho từng cây con (cây con của BST cũng là BST) → toàn bộ dãy tăng dần.

Ví dụ cây T: in-order = (in-order cây con gốc `3`) + `[8]` + (in-order cây con gốc `13`) = `[1, 3, 4, 6, 7]` + `[8]` + `[10, 12, 13, 15]`.

Hệ quả hay dùng: **kiểm tra một cây có phải BST** ⇔ in-order của nó **tăng ngặt**.

In [7]:
def inorder_list(root):
    out = []
    inorder(root, out)
    return out


def is_bst(root):
    vals = inorder_list(root)
    return all(vals[i] < vals[i + 1] for i in range(len(vals) - 1))


bad = build_tree([5, 3, 8, 1, 6])
print_tree(bad)
print("in-order:", inorder_list(bad), "-> BST?", is_bst(bad))   # 6 đứng trước 5 -> không tăng
print()
print("in-order Example 1:", inorder_list(root_ex1), "-> BST?", is_bst(root_ex1))
print("in-order cây T    :", inorder_list(root_T), "-> BST?", is_bst(root_T))

  _5
 /  \
 3  8
/ \
1 6
in-order: [1, 3, 6, 5, 8] -> BST? False

in-order Example 1: [1, 2, 3, 4, 5] -> BST? True
in-order cây T    : [1, 3, 4, 6, 7, 8, 10, 12, 13, 15] -> BST? True


### 4.3 Tìm kiếm trên BST: mỗi bước bỏ đi cả một cây con

Muốn biết giá trị `x` có trong BST không, ta **không** cần duyệt cả cây. Đứng ở node hiện tại:

- `x == node.val` → thấy rồi;
- `x < node.val` → `x` (nếu có) chắc chắn nằm bên **trái** (vì bên phải toàn số lớn hơn `node.val`) → đi trái;
- `x > node.val` → đi phải.

Giống hệt **binary search** trên mảng đã sắp xếp: mỗi bước loại bỏ cả một nửa (một cây con). Số bước tối đa = số tầng = `h`, nên tìm kiếm tốn `O(h)`.

| Hình dạng cây | `n = 7` | `n = 10^4` |
|---|:---:|:---:|
| cân bằng | `h = 3` | `h = 14` |
| lệch thành "dây" | `h = 7` | `h = 10^4` |

Đó là lý do câu follow-up nói "giả sử BST **cân bằng**": khi đó `O(h) = O(log n)` rất nhỏ.

Trên cây T: tìm `7` đi `8 → 3 → 6 → 7` (thấy); tìm `9` đi `8 → 13 → 10` rồi rơi vào `None` (không có).

In [8]:
def search_bst(root, x):
    node, path = root, []
    while node is not None:
        path.append(node.val)
        if x == node.val:
            print(f"tìm {x:>2}: đường đi {path} -> THẤY")
            return node
        node = node.left if x < node.val else node.right
    print(f"tìm {x:>2}: đường đi {path} -> KHÔNG có")
    return None


search_bst(root_T, 7)
search_bst(root_T, 9)
search_bst(root_T, 12)
search_bst(root_T, 0)

tìm  7: đường đi [8, 3, 6, 7] -> THẤY
tìm  9: đường đi [8, 13, 10] -> KHÔNG có
tìm 12: đường đi [8, 13, 10, 12] -> THẤY
tìm  0: đường đi [8, 3, 1] -> KHÔNG có


### 4.4 Helper tạo BST: chèn từng giá trị, hoặc dựng cây cân bằng

Ta cần thêm hai cách tạo BST để kiểm thử về sau:

1. `bst_insert(root, val)`: đi đúng như khi **tìm** `val`; tới chỗ `None` thì gắn node mới vào đó. `build_bst(values)` chèn lần lượt các giá trị. **Thứ tự chèn quyết định hình dạng**: chèn `4,2,5,1,3` ra đúng cây Example 1, còn chèn `1,2,3,4,5` (tăng dần) ra một **sợi dây** lệch phải, `h = n`. Viết bằng vòng lặp để chèn được cả vào cây rất sâu.
2. `build_balanced_bst(sorted_vals)`: lấy phần tử **giữa** của dãy đã sắp xếp làm root, nửa trái dựng cây con trái, nửa phải dựng cây con phải (đệ quy). Hai nửa lệch nhau tối đa 1 phần tử → cây **cân bằng**, `h ≈ log₂ n`. Tham số `node_cls` cho phép dựng bằng một loại node khác (sẽ dùng ở mục 12.2 để **đếm** số node bị đọc).

In [9]:
def bst_insert(root, val):
    # Chèn val (giả sử chưa có trong cây) vào BST, trả về root.
    new = TreeNode(val)
    if root is None:
        return new
    node = root
    while True:
        if val < node.val:
            if node.left is None:
                node.left = new
                return root
            node = node.left
        else:
            if node.right is None:
                node.right = new
                return root
            node = node.right


def build_bst(values):
    root = None
    for v in values:
        root = bst_insert(root, v)
    return root


def build_balanced_bst(sorted_vals, node_cls=TreeNode):
    # Phần tử giữa làm root -> hai nửa chênh nhau tối đa 1 -> cây cân bằng.
    if not sorted_vals:
        return None
    mid = len(sorted_vals) // 2
    return node_cls(sorted_vals[mid],
                    build_balanced_bst(sorted_vals[:mid], node_cls),
                    build_balanced_bst(sorted_vals[mid + 1:], node_cls))


print("Chèn theo thứ tự 4,2,5,1,3:")
print_tree(build_bst([4, 2, 5, 1, 3]))
print("\nChèn theo thứ tự tăng dần 1,2,3,4,5 -> cây lệch thành 'dây':")
print_tree(build_bst([1, 2, 3, 4, 5]))
print("\nbuild_balanced_bst([1..7]):")
print_tree(build_balanced_bst([1, 2, 3, 4, 5, 6, 7]))
print("\nchiều cao cây cân bằng 10^4 node:", height(build_balanced_bst(list(range(10_000)))))
print("chiều cao cây 'dây' 500 node     :", height(build_bst(range(500))))

Chèn theo thứ tự 4,2,5,1,3:
  _4
 /  \
 2  5
/ \
1 3

Chèn theo thứ tự tăng dần 1,2,3,4,5 -> cây lệch thành 'dây':
1
 \
 2
  \
  3
   \
   4
    \
    5

build_balanced_bst([1..7]):
  _4_
 /   \
 2   6
/ \ / \
1 3 5 7

chiều cao cây cân bằng 10^4 node: 14
chiều cao cây 'dây' 500 node     : 500


## 5. Ôn bài #270: tìm **một** giá trị gần `target` nhất

### 5.1 Đề #270 (Closest Binary Search Tree Value, Easy — cũng là bài Premium)

> Cho `root` của một BST và `target`, trả về **giá trị** trong BST gần `target` nhất. Nếu có nhiều đáp án, trả về giá trị **nhỏ nhất**.
>
> Ví dụ: `root = [4,2,5,1,3]`, `target = 3.714286` → `4`.

Bài 272 chính là bản "nâng cấp" của 270: thay vì 1 giá trị, cần `k` giá trị. Học kỹ 270 trước vì **ý tưởng của nó là nửa đầu của lời giải chính**.

**Ý tưởng:** đi xuống từ root **giống như đang tìm `target`** (target < node thì sang trái, ngược lại sang phải). Trên đường đi, giá trị nào gần `target` hơn `best` thì cập nhật `best`. Chỉ đi **một đường** → `O(h)`.

Câu hỏi tự nhiên của người mới: *"Chỉ nhìn các node trên một đường đi, sao chắc không bỏ sót node gần hơn ở nhánh khác?"* — mục 5.2 trả lời. Trước hết xem nó chạy:

In [10]:
def closest_value(root, target, verbose=False):
    best = root.val
    node = root
    while node is not None:
        d_node, d_best = abs(node.val - target), abs(best - target)
        if d_node < d_best or (d_node == d_best and node.val < best):   # hòa -> lấy số nhỏ hơn
            best = node.val
        if verbose:
            way = "trái" if target < node.val else "phải"
            print(f"  thăm {node.val:>2}: |{node.val} - {target}| = {d_node:.4f}  best = {best:>2}  -> sang {way}")
        node = node.left if target < node.val else node.right
    return best


print("Example #270, target = 3.714286")
print("  => đáp án:", closest_value(root_ex1, 3.714286, verbose=True))
print("Cây T, target = 9.1")
print("  => đáp án:", closest_value(root_T, 9.1, verbose=True))

Example #270, target = 3.714286
  thăm  4: |4 - 3.714286| = 0.2857  best =  4  -> sang trái
  thăm  2: |2 - 3.714286| = 1.7143  best =  4  -> sang phải
  thăm  3: |3 - 3.714286| = 0.7143  best =  4  -> sang phải
  => đáp án: 4
Cây T, target = 9.1
  thăm  8: |8 - 9.1| = 1.1000  best =  8  -> sang phải
  thăm 13: |13 - 9.1| = 3.9000  best =  8  -> sang trái
  thăm 10: |10 - 9.1| = 0.9000  best = 10  -> sang trái
  => đáp án: 10


### 5.2 Predecessor và successor của `target` — vì sao một đường đi là đủ

Sắp các giá trị của cây T tăng dần: `[1, 3, 4, 6, 7, 8, 10, 12, 13, 15]`. Với `target = 9.1`, nó "rơi" vào giữa `8` và `10`. Định nghĩa (theo đúng quy ước dùng cho cả notebook):

- **predecessor** (`pred`, "kẻ đứng trước") = giá trị **lớn nhất** mà **`< target`** → ở đây là `8`;
- **successor** (`succ`, "kẻ đứng sau") = giá trị **nhỏ nhất** mà **`>= target`** → ở đây là `10`.

(Giá trị **bằng** `target` được xếp về phía `succ`, để mỗi giá trị thuộc **đúng một** phía.)

**Giá trị gần nhất chắc chắn là `pred` hoặc `succ`:** mọi giá trị khác nằm "xa hơn" một trong hai. Ví dụ `7 < 8 < 9.1` nên `7` xa `target` hơn `8`; `12 > 10 >= 9.1` nên `12` xa hơn `10`.

**Và cả `pred` lẫn `succ` đều nằm trên đường tìm kiếm.** Lý do: lấy một giá trị `v < target` bất kỳ **không** nằm trên đường đi. Đường đi và `v` phải "tách nhau" tại một node `a` nào đó trên đường:

- nếu `v` ở bên **trái** của `a` còn đường đi rẽ **phải** tại `a` → `a < target` (nên mới rẽ phải) và `v < a` → `a` là ứng viên `pred` **tốt hơn** `v`, mà `a` thì nằm trên đường;
- nếu `v` ở bên **phải** của `a` còn đường đi rẽ **trái** tại `a` → `v > a >= target`, mâu thuẫn với `v < target`.

Vậy ứng viên `pred` tốt nhất luôn nằm trên đường đi (lập luận đối xứng cho `succ`). Cụ thể hơn, khi đi xuống:

- mỗi lần **rẽ phải** tại node `x` (vì `x < target`): `x` là một ứng viên `pred`; node rẽ phải **cuối cùng** chính là `pred`;
- mỗi lần **rẽ trái** tại node `x` (vì `x >= target`): `x` là ứng viên `succ`; node rẽ trái **cuối cùng** chính là `succ`.

| Node trên đường (target = 9.1) | So sánh | Rẽ | Xếp vào nhóm |
|:---:|---|:---:|:---:|
| 8 | 8 < 9.1 | phải | `< target` |
| 13 | 13 >= 9.1 | trái | `>= target` |
| 10 | 10 >= 9.1 | trái | `>= target` |
| (None) | dừng | | |

Nhóm `< target` = `[8]` → `pred = 8`; nhóm `>= target` = `[13, 10]` → `succ = 10`. Hai danh sách nhỏ này sẽ trở thành **hai stack** của cách chính (mục 9) — hãy nhớ chúng!

In [11]:
def path_split(root, target):
    smaller, bigger = [], []        # node rẽ phải (< target) / node rẽ trái (>= target)
    node = root
    while node is not None:
        if node.val < target:
            smaller.append(node.val)
            node = node.right
        else:
            bigger.append(node.val)
            node = node.left
    return smaller, bigger


all_vals = inorder_list(root_T)
for tgt in [9.1, 6, 11.5, 0.5, 20]:
    s, b = path_split(root_T, tgt)
    pred = s[-1] if s else None
    succ = b[-1] if b else None
    print(f"target={tgt:<4}  nhóm '<': {str(s):<12} nhóm '>=': {str(b):<10} -> pred={pred}, succ={succ}")
    # đối chiếu với định nghĩa trên dãy đã sắp xếp
    assert pred == max((v for v in all_vals if v < tgt), default=None)
    assert succ == min((v for v in all_vals if v >= tgt), default=None)
print("pred/succ lấy từ đường đi khớp định nghĩa")

target=9.1   nhóm '<': [8]          nhóm '>=': [13, 10]   -> pred=8, succ=10
target=6     nhóm '<': [3, 4]       nhóm '>=': [8, 6]     -> pred=4, succ=6
target=11.5  nhóm '<': [8, 10]      nhóm '>=': [13, 12]   -> pred=10, succ=12
target=0.5   nhóm '<': []           nhóm '>=': [8, 3, 1]  -> pred=None, succ=1
target=20    nhóm '<': [8, 13, 15]  nhóm '>=': []         -> pred=15, succ=None
pred/succ lấy từ đường đi khớp định nghĩa


## 6. Cách 1 — Gom hết giá trị rồi sắp xếp theo khoảng cách

### 6.1 Ý tưởng người mới tự nghĩ ra

"Muốn `k` giá trị gần nhất? Lấy **hết** các giá trị ra một list, **sắp xếp theo khoảng cách** tới `target`, rồi lấy `k` phần tử đầu."

Đây chính là cách ta làm bằng tay ở bảng mục 1.1. Cần một công cụ mới: **sắp xếp theo một tiêu chí tùy chọn**.

### 6.2 `sorted(..., key=...)` và `lambda`

- `sorted(a)` trả về list mới, sắp **theo giá trị** tăng dần.
- `sorted(a, key=f)` sắp theo **`f(x)`** thay vì `x`: Python tính `f(x)` cho từng phần tử, rồi xếp theo các con số đó. `a.sort(key=f)` làm y hệt nhưng sửa trực tiếp `a`.
- `lambda x: abs(x - target)` là một **hàm ngắn không tên**: nhận `x`, trả về `|x - target|`. Viết `key=lambda x: abs(x - target)` nghĩa là "xếp theo khoảng cách tới target".

Ví dụ: `sorted([-7, 2, -1, 5], key=abs)` → so `|−7| = 7, |2| = 2, |−1| = 1, |5| = 5` → kết quả `[-1, 2, 5, -7]`.

In [12]:
nums = [-7, 2, -1, 5]
print(sorted(nums))                     # theo giá trị: [-7, -1, 2, 5]
print(sorted(nums, key=abs))            # theo |x|    : [-1, 2, 5, -7]

dist = lambda x: abs(x - 3.714286)      # lambda = hàm ngắn không tên
print(round(dist(4), 6), round(dist(3), 6))      # 0.285714 0.714286
print(sorted([4, 2, 5, 1, 3], key=dist))         # gần 3.714286 nhất đứng đầu: [4, 3, 5, 2, 1]

[-7, -1, 2, 5]
[-1, 2, 5, -7]
0.285714 0.714286
[4, 3, 5, 2, 1]


In [13]:
def closest_k_sort(root, target, k):
    vals = inorder_list(root)                        # lấy hết giá trị (thứ tự nào cũng được)
    vals.sort(key=lambda x: abs(x - target))         # gần target nhất đứng đầu
    return vals[:k]                                  # k phần tử đầu


print(closest_k_sort(build_tree([4, 2, 5, 1, 3]), 3.714286, 2))   # Example 1 -> [4, 3]
print(closest_k_sort(build_tree([1]), 0.0, 1))                    # Example 2 -> [1]
print(closest_k_sort(root_T, 9.1, 4))                             # cây T     -> [10, 8, 7, 12]

[4, 3]
[1]
[10, 8, 7, 12]


### 6.3 Độ phức tạp và nút thắt của cách 1

- **Time** `O(n log n)`: duyệt cây `O(n)` + sắp xếp `n` phần tử `O(n log n)`.
- **Space** `O(n)`: list chứa toàn bộ giá trị.

Hai điểm "phí":

1. Ta **sắp xếp cả `n` phần tử** trong khi chỉ cần `k` phần tử tốt nhất. Với `n = 10^4`, `k = 3`, ta xếp thứ tự cho 10 000 số chỉ để lấy 3 số.
2. Ta **không dùng chút nào tính chất BST** — cách 1 chạy y hệt trên một cây nhị phân bất kỳ. (Điểm này sẽ được giải quyết ở cách 3 và 4.)

Sửa điểm 1 trước: thay vì sắp xếp tất cả, chỉ **giữ `k` ứng viên tốt nhất đến thời điểm hiện tại**. Công cụ cho việc này là **heap**.

## 7. Cách 2 — Max-heap kích thước `k`

### 7.1 Heap và module `heapq` từ căn bản

**Heap** (đống) là cấu trúc luôn cho ta **phần tử nhỏ nhất** thật nhanh. Python có module `heapq` làm việc trên một `list` thường:

| Lệnh | Tác dụng | Chi phí (heap có `m` phần tử) |
|---|---|:---:|
| `heapq.heappush(h, x)` | thêm `x` | `O(log m)` |
| `heapq.heappop(h)` | lấy ra **và xóa** phần tử nhỏ nhất | `O(log m)` |
| `h[0]` | **xem** phần tử nhỏ nhất (không xóa) | `O(1)` |

`heapq` chỉ có **min-heap** (nhỏ nhất ở đỉnh). Muốn **max-heap** (lớn nhất ở đỉnh) thì dùng mẹo **đổi dấu**: đẩy `-x` vào; số lớn nhất có `-x` nhỏ nhất nên nằm ở đỉnh.

Phần tử của heap có thể là **tuple**: Python so sánh tuple theo phần tử đầu trước, bằng nhau mới so phần tử thứ hai. Nên `(-khoảng_cách, giá_trị)` được xếp theo khoảng cách.

In [14]:
h = []
for x in [5, 1, 8, 3]:
    heapq.heappush(h, x)
    print(f"push {x}: phần tử nhỏ nhất h[0] = {h[0]}")
print("heappop ->", heapq.heappop(h), "| nhỏ nhất còn lại:", h[0])

# Max-heap bằng cách đổi dấu: số LỚN nhất có -x NHỎ nhất
mh = []
for x in [5, 1, 8, 3]:
    heapq.heappush(mh, -x)
print("lớn nhất =", -mh[0])                       # 8
print("lấy ra số lớn nhất ->", -heapq.heappop(mh))  # 8

# Tuple so theo phần tử đầu trước
print((-2.5, 7) < (-1.0, 3))                      # True vì -2.5 < -1.0

push 5: phần tử nhỏ nhất h[0] = 5
push 1: phần tử nhỏ nhất h[0] = 1
push 8: phần tử nhỏ nhất h[0] = 1
push 3: phần tử nhỏ nhất h[0] = 1
heappop -> 1 | nhỏ nhất còn lại: 3
lớn nhất = 8
lấy ra số lớn nhất -> 8
True


### 7.2 Ý tưởng: giữ đúng `k` ứng viên, đá phần tử **xa nhất** ra

Duyệt lần lượt mọi giá trị `x`. Heap chứa **`k` giá trị gần nhất trong số đã xét**:

1. Đẩy `x` vào heap.
2. Nếu heap có `k + 1` phần tử → bỏ đi phần tử **xa `target` nhất** (nó chắc chắn không thuộc top `k` của những gì đã xét).

Muốn lấy ngay phần tử **xa nhất** → cần **max-heap theo khoảng cách** → lưu `(-|x - target|, x)`. Phần tử ở đỉnh `h[0]` có `-khoảng_cách` nhỏ nhất, tức khoảng cách **lớn nhất**, đúng thứ cần đá.

Ví dụ cây T, `target = 9.1`, `k = 3`: các giá trị đi theo thứ tự in-order `1, 3, 4, 6, ...`. Khi tới `6`, heap đang có `{1, 3, 4}` (đủ 3), thêm `6` thành 4 phần tử → đá `1` (xa nhất, `8.1`). Khi tới `12` (khoảng cách `2.9`), heap đang là `{7, 8, 10}` với phần tử xa nhất là `7` (`2.1`); thêm `12` vào rồi đá phần tử xa nhất — chính là **`12` vừa vào**.

Cell dưới có hai hàm: `closest_k_heap` (bản sạch, duyệt cây bằng đệ quy in-order, không tạo list trung gian) và `trace_heap` (bản in trạng thái từng bước).

In [15]:
def closest_k_heap(root, target, k):
    heap = []                                        # chứa (-khoảng cách, giá trị), tối đa k phần tử

    def visit(node):
        if node is None:
            return
        visit(node.left)
        heapq.heappush(heap, (-abs(node.val - target), node.val))
        if len(heap) > k:
            heapq.heappop(heap)                      # đá phần tử XA nhất (đỉnh max-heap)
        visit(node.right)

    visit(root)
    return [x for _, x in heap]


def trace_heap(root, target, k):
    heap = []
    for x in inorder_list(root):
        heapq.heappush(heap, (-abs(x - target), x))
        kicked = heapq.heappop(heap)[1] if len(heap) > k else None
        msg = f"đá {kicked} ra" if kicked is not None else "chưa đủ k, giữ"
        kept = sorted(v for _, v in heap)
        print(f"x={x:>2}  |x-t|={abs(x - target):.1f}  {msg:<15} heap={str(kept):<12} xa nhất={heap[0][1]}")
    return [x for _, x in heap]


print("Example 1:", closest_k_heap(root_ex1, 3.714286, 2))
print("\nTrace cây T, target = 9.1, k = 3:")
print("Kết quả:", sorted(trace_heap(root_T, 9.1, 3)))

Example 1: [3, 4]

Trace cây T, target = 9.1, k = 3:
x= 1  |x-t|=8.1  chưa đủ k, giữ  heap=[1]          xa nhất=1
x= 3  |x-t|=6.1  chưa đủ k, giữ  heap=[1, 3]       xa nhất=1
x= 4  |x-t|=5.1  chưa đủ k, giữ  heap=[1, 3, 4]    xa nhất=1
x= 6  |x-t|=3.1  đá 1 ra         heap=[3, 4, 6]    xa nhất=3
x= 7  |x-t|=2.1  đá 3 ra         heap=[4, 6, 7]    xa nhất=4
x= 8  |x-t|=1.1  đá 4 ra         heap=[6, 7, 8]    xa nhất=6
x=10  |x-t|=0.9  đá 6 ra         heap=[7, 8, 10]   xa nhất=7
x=12  |x-t|=2.9  đá 12 ra        heap=[7, 8, 10]   xa nhất=7
x=13  |x-t|=3.9  đá 13 ra        heap=[7, 8, 10]   xa nhất=7
x=15  |x-t|=5.9  đá 15 ra        heap=[7, 8, 10]   xa nhất=7
Kết quả: [7, 8, 10]


### 7.3 Độ phức tạp và nút thắt của cách 2

- **Time** `O(n log k)`: mỗi giá trị một lần `heappush` (+ có thể một `heappop`) trên heap tối đa `k + 1` phần tử.
- **Space** `O(k + h)`: heap `k` phần tử + đệ quy sâu tối đa `h` tầng.

Tốt hơn cách 1 khi `k` nhỏ. Nhưng vẫn còn hai điểm phí:

1. Vẫn **đọc cả `n` node**.
2. Vẫn **không dùng tính chất BST**. Trong trace trên, các giá trị đến theo thứ tự **tăng dần** (`1, 3, 4, ...`) mà heap không hề tận dụng.

## 8. Cách 3 — Khai thác "in-order = dãy tăng dần"

### 8.1 Quan sát then chốt: `k` giá trị gần nhất là một **đoạn liền** của dãy tăng dần

Dãy in-order của cây T: `[1, 3, 4, 6, 7, 8, 10, 12, 13, 15]`, `target = 9.1`, `k = 4` → đáp án `{7, 8, 10, 12}` nằm ở các chỉ số `4, 5, 6, 7` — **liền nhau**, bao quanh chỗ `target` "rơi" vào (giữa `8` và `10`).

**Vì sao luôn liền nhau?** Giả sử `x` thuộc đáp án, và `y` là một giá trị nằm **giữa** `x` và `target`. Khi đó `|y − target| < |x − target|` (`y` gần hơn `x`). Nếu `y` **không** thuộc đáp án, ta đổi `x` lấy `y` sẽ được một tập **gần hơn** — vô lý với việc đáp án là tập gần nhất (và duy nhất). Vậy **mọi giá trị nằm giữa một phần tử của đáp án và `target` cũng thuộc đáp án** → đáp án là một đoạn liền chứa vị trí của `target`.

Nhờ vậy ta không còn phải "chọn bất kỳ `k` phần tử nào" mà chỉ cần **mở rộng một đoạn** từ vị trí của `target` ra hai phía.

In [16]:
arr = inorder_list(root_T)
target, k = 9.1, 4
chosen = set(closest_k_sort(root_T, target, k))
for i, x in enumerate(arr):
    mark = "<- chọn" if x in chosen else ""
    print(f"i={i}  x={x:>2}  |x - t| = {abs(x - target):.1f}  {mark}")
idx = [i for i, x in enumerate(arr) if x in chosen]
print("các chỉ số được chọn:", idx, "-> liền nhau?", idx == list(range(idx[0], idx[-1] + 1)))

i=0  x= 1  |x - t| = 8.1  
i=1  x= 3  |x - t| = 6.1  
i=2  x= 4  |x - t| = 5.1  
i=3  x= 6  |x - t| = 3.1  
i=4  x= 7  |x - t| = 2.1  <- chọn
i=5  x= 8  |x - t| = 1.1  <- chọn
i=6  x=10  |x - t| = 0.9  <- chọn
i=7  x=12  |x - t| = 2.9  <- chọn
i=8  x=13  |x - t| = 3.9  
i=9  x=15  |x - t| = 5.9  
các chỉ số được chọn: [4, 5, 6, 7] -> liền nhau? True


### 8.2 `bisect_left`: tìm chỗ `target` "rơi" vào trong `O(log n)`

`bisect_left(a, x)` (module `bisect`) nhận một list **đã sắp xếp** `a` và trả về **chỉ số `i` nhỏ nhất sao cho `a[i] >= x`** — tức vị trí nếu muốn chèn `x` vào mà vẫn giữ thứ tự. Cũng chính là **số phần tử `< x`**.

Với `arr = [1, 3, 4, 6, 7, 8, 10, 12, 13, 15]`:

| x | `bisect_left(arr, x)` | `arr[i-1]` (lớn nhất `< x`) | `arr[i]` (nhỏ nhất `>= x`) |
|---:|:---:|:---:|:---:|
| 9.1 | 6 | 8 | 10 |
| 8 | 5 | 7 | 8 (bằng `x` → phía phải) |
| 0.5 | 0 | (không có) | 1 |
| 20 | 10 | 15 | (không có) |

Nhìn cột 3 và 4: chính là **pred** và **succ** của mục 5.2! Bên trong, `bisect_left` là **binary search**: mỗi bước so phần tử giữa rồi bỏ đi một nửa, nên tốn `O(log n)`. Cell dưới tự cài lại để thấy không có gì "phép thuật".

In [17]:
arr = [1, 3, 4, 6, 7, 8, 10, 12, 13, 15]
for x in [9.1, 8, 0.5, 20]:
    i = bisect_left(arr, x)
    left = arr[i - 1] if i > 0 else None
    right = arr[i] if i < len(arr) else None
    print(f"bisect_left(arr, {x:>4}) = {i:>2}  -> lớn nhất < x: {left},  nhỏ nhất >= x: {right}")


def my_bisect_left(a, x):
    lo, hi = 0, len(a)             # đáp án luôn nằm trong đoạn [lo, hi]
    while lo < hi:
        mid = (lo + hi) // 2
        if a[mid] < x:
            lo = mid + 1           # a[mid] < x: vị trí cần tìm ở bên phải mid
        else:
            hi = mid               # a[mid] >= x: mid có thể là đáp án, thu hẹp về bên trái
    return lo


assert all(my_bisect_left(arr, x) == bisect_left(arr, x) for x in [-1, 0.5, 1, 2, 8, 9.1, 15, 20])
print("my_bisect_left khớp bisect_left trên mọi x thử")

bisect_left(arr,  9.1) =  6  -> lớn nhất < x: 8,  nhỏ nhất >= x: 10
bisect_left(arr,    8) =  5  -> lớn nhất < x: 7,  nhỏ nhất >= x: 8
bisect_left(arr,  0.5) =  0  -> lớn nhất < x: None,  nhỏ nhất >= x: 1
bisect_left(arr,   20) = 10  -> lớn nhất < x: 15,  nhỏ nhất >= x: None
my_bisect_left khớp bisect_left trên mọi x thử


### 8.3 Hai con trỏ mở rộng từ vị trí của `target`

Có `i = bisect_left(arr, target)`. Đặt hai con trỏ:

- `lo = i - 1`: trỏ vào **pred** (giá trị lớn nhất `< target`), sẽ đi dần sang **trái**;
- `hi = i`: trỏ vào **succ** (giá trị nhỏ nhất `>= target`), sẽ đi dần sang **phải**.

Lặp `k` lần, mỗi lần "nhận" một giá trị vào đoạn đáp án:

- nếu bên trái hết (`lo < 0`) → nhận bên phải: `hi += 1`;
- nếu bên phải hết (`hi >= n`) → nhận bên trái: `lo -= 1`;
- ngược lại so `target - arr[lo]` với `arr[hi] - target`: bên nào **nhỏ hơn** (gần hơn) thì nhận bên đó.

Không cần `abs` vì `arr[lo] < target <= arr[hi]` nên cả hai hiệu đều `>= 0`. Hòa thì nhận bên trái (`<=`) — với đề 272 điều này không ảnh hưởng tập đáp án (mục 9.3 giải thích kỹ).

Sau `k` lần, đáp án chính là **đoạn `arr[lo+1 : hi]`** (từ `lo + 1` đến `hi - 1`).

Bảng trace cho `arr` của cây T, `target = 9.1`, `k = 4` (ban đầu `i = 6`, `lo = 5`, `hi = 6`):

| Bước | `lo` (`arr[lo]`) | `hi` (`arr[hi]`) | `t − arr[lo]` | `arr[hi] − t` | Nhận | Đoạn `arr[lo+1:hi]` sau bước |
|:---:|:---:|:---:|:---:|:---:|---|---|
| 1 | 5 (8) | 6 (10) | 1.1 | 0.9 | `10`, `hi → 7` | `[10]` |
| 2 | 5 (8) | 7 (12) | 1.1 | 2.9 | `8`, `lo → 4` | `[8, 10]` |
| 3 | 4 (7) | 7 (12) | 2.1 | 2.9 | `7`, `lo → 3` | `[7, 8, 10]` |
| 4 | 3 (6) | 7 (12) | 3.1 | 2.9 | `12`, `hi → 8` | `[7, 8, 10, 12]` |

In [18]:
def closest_k_two_pointers(arr, target, k):
    # arr tăng dần. Trả về k giá trị gần target nhất = một đoạn liền của arr.
    hi = bisect_left(arr, target)          # arr[hi] = nhỏ nhất >= target
    lo = hi - 1                            # arr[lo] = lớn nhất  <  target
    for _ in range(k):
        if lo < 0:                                     # bên trái hết hàng
            hi += 1
        elif hi >= len(arr):                           # bên phải hết hàng
            lo -= 1
        elif target - arr[lo] <= arr[hi] - target:     # bên trái gần hơn (hoặc bằng)
            lo -= 1
        else:                                          # bên phải gần hơn
            hi += 1
    return arr[lo + 1:hi]


def closest_k_bisect(root, target, k):
    return closest_k_two_pointers(inorder_list(root), target, k)


def trace_two_pointers(arr, target, k):
    hi = bisect_left(arr, target)
    lo = hi - 1
    print(f"bisect_left = {hi}: lo = {lo}, hi = {hi}")
    for step in range(1, k + 1):
        dl = f"{target - arr[lo]:.1f}" if lo >= 0 else "hết"
        dr = f"{arr[hi] - target:.1f}" if hi < len(arr) else "hết"
        if lo < 0 or (hi < len(arr) and target - arr[lo] > arr[hi] - target):
            took, hi = arr[hi], hi + 1
        else:
            took, lo = arr[lo], lo - 1
        print(f"Bước {step}: t-arr[lo]={dl:<4} arr[hi]-t={dr:<4} -> nhận {took:>2}   đoạn = {arr[lo + 1:hi]}")
    return arr[lo + 1:hi]


print(closest_k_bisect(root_ex1, 3.714286, 2))     # [3, 4]
print(closest_k_bisect(build_tree([1]), 0.0, 1))   # [1]
print(closest_k_bisect(root_T, 14.6, 4))           # [10, 12, 13, 15]
print()
_ = trace_two_pointers(inorder_list(root_T), 9.1, 4)

[3, 4]
[1]
[10, 12, 13, 15]

bisect_left = 6: lo = 5, hi = 6
Bước 1: t-arr[lo]=1.1  arr[hi]-t=0.9  -> nhận 10   đoạn = [10]
Bước 2: t-arr[lo]=1.1  arr[hi]-t=2.9  -> nhận  8   đoạn = [8, 10]
Bước 3: t-arr[lo]=2.1  arr[hi]-t=2.9  -> nhận  7   đoạn = [7, 8, 10]
Bước 4: t-arr[lo]=3.1  arr[hi]-t=2.9  -> nhận 12   đoạn = [7, 8, 10, 12]


### 8.4 Độ phức tạp và nút thắt của cách 3a (bisect + hai con trỏ)

- Dựng mảng in-order: `O(n)` time, `O(n)` space.
- Sau khi có mảng: `bisect_left` `O(log n)` + `k` bước con trỏ `O(k)` → `O(log n + k)`.
- Tổng: `O(n)` time, `O(n)` space. (Nếu phải trả lời **rất nhiều** truy vấn trên cùng một cây, dựng mảng một lần rồi mỗi truy vấn chỉ tốn `O(log n + k)` — rất đáng.)

Nút thắt: ta dựng **cả mảng `n` phần tử**, trong khi chỉ có `k` giá trị quanh `target` là quan trọng. Ý tưởng tiếp theo: **vừa duyệt in-order vừa giữ một cửa sổ `k` phần tử, và dừng ngay khi biết chắc không còn gì tốt hơn**. Muốn "dừng giữa chừng" một phép duyệt thì đệ quy hơi vướng; ta cần duyệt in-order bằng **vòng lặp + stack tự quản lý**. Đây cũng là nền móng trực tiếp của cách 4.

### 8.5 Stack và duyệt in-order bằng vòng lặp

**Stack** (ngăn xếp) ngược với queue: **vào sau, ra trước** (LIFO — Last In, First Out), như chồng đĩa. Python dùng luôn `list`: `append` đặt lên đỉnh, `pop()` lấy ở đỉnh, `st[-1]` xem đỉnh — đều `O(1)`.

In-order = "trái → node → phải". Bằng vòng lặp:

```text
node = root
lặp khi stack còn phần tử hoặc node khác None:
    1) đi TRÁI hết cỡ, đẩy mọi node gặp dọc đường vào stack
    2) node = stack.pop()       # node nhỏ nhất chưa thăm
    3) thăm node
    4) node = node.right        # chuyển sang cây con phải (vòng sau lại đi trái hết cỡ từ đó)
```

Vì sao đúng? Các node trong stack là những node **đang chờ** cây con trái của mình được thăm xong. Đi trái hết cỡ rồi `pop` → lấy được node nhỏ nhất chưa thăm. Thăm xong node thì tới lượt cây con phải của nó — đúng thứ tự "trái → node → phải".

Hãy để ý bước 1 + bước 4: **"pop một node, rồi đẩy chuỗi trái của cây con phải"**. Câu này sẽ xuất hiện nguyên văn trong cách 4.

In [19]:
st = []
st.append("A")
st.append("B")
st.append("C")
print("stack:", st, "| đỉnh:", st[-1])
print("pop ->", st.pop(), "| còn:", st)        # C vào sau nên ra trước


def inorder_iterative(root, verbose=False):
    out, stack, node = [], [], root
    while stack or node is not None:
        while node is not None:              # 1) đi trái hết cỡ, đẩy dọc đường
            stack.append(node)
            node = node.left
        node = stack.pop()                   # 2) node nhỏ nhất chưa thăm
        out.append(node.val)                 # 3) thăm
        if verbose:
            print(f"thăm {node.val:>2}   stack còn {[n.val for n in stack]}")
        node = node.right                    # 4) sang cây con phải
    return out


print(inorder_iterative(root_T, verbose=True))
assert inorder_iterative(root_T) == inorder_list(root_T)

stack: ['A', 'B', 'C'] | đỉnh: C
pop -> C | còn: ['A', 'B']
thăm  1   stack còn [8, 3]
thăm  3   stack còn [8]
thăm  4   stack còn [8, 6]
thăm  6   stack còn [8]
thăm  7   stack còn [8]
thăm  8   stack còn []
thăm 10   stack còn [13]
thăm 12   stack còn [13]
thăm 13   stack còn []
thăm 15   stack còn []
[1, 3, 4, 6, 7, 8, 10, 12, 13, 15]


### 8.6 Cách 3b — Cửa sổ trượt bằng `deque` trong lúc duyệt in-order, dừng sớm

Các giá trị đến theo thứ tự **tăng dần**. Giữ một `deque` tên `window` chứa tối đa `k` giá trị **liên tiếp** (theo 8.1, đáp án là một đoạn liền). Với mỗi giá trị mới `x`:

1. Nếu `window` chưa đủ `k` phần tử → `append(x)`.
2. Nếu đã đủ `k` và `x` **gần hơn** phần tử **trái nhất** `window[0]` → `popleft()` rồi `append(x)`: cửa sổ **trượt** sang phải một ô.
3. Ngược lại → **`break`**, dừng hẳn việc duyệt.

**Vì sao so với `window[0]`?** `x` lớn hơn mọi phần tử trong cửa sổ. Cửa sổ mới vẫn phải là đoạn liền, nên lựa chọn duy nhất là "bỏ phần tử trái nhất, thêm `x`" hoặc "giữ nguyên". So `x` với `window[0]` là đủ.

**Vì sao `break` an toàn?** Nếu `x` **không** gần hơn `window[0]` thì chắc chắn `x >= target` (nếu `x < target` thì `window[0] < x < target`, tức `x` gần hơn — mâu thuẫn). Các giá trị đến sau còn lớn hơn `x` nên còn **xa hơn nữa** → không ai cải thiện được cửa sổ.

Trace dự đoán trên cây T, `target = 9.1`, `k = 4`:

| x | \|x − t\| | \|window[0] − t\| | Hành động | window sau |
|---:|---:|---:|---|---|
| 1 | 8.1 | — | chưa đủ k → thêm | `[1]` |
| 3 | 6.1 | — | thêm | `[1, 3]` |
| 4 | 5.1 | — | thêm | `[1, 3, 4]` |
| 6 | 3.1 | — | thêm | `[1, 3, 4, 6]` |
| 7 | 2.1 | 8.1 (của 1) | gần hơn → bỏ 1, thêm 7 | `[3, 4, 6, 7]` |
| 8 | 1.1 | 6.1 (của 3) | bỏ 3, thêm 8 | `[4, 6, 7, 8]` |
| 10 | 0.9 | 5.1 (của 4) | bỏ 4, thêm 10 | `[6, 7, 8, 10]` |
| 12 | 2.9 | 3.1 (của 6) | bỏ 6, thêm 12 | `[7, 8, 10, 12]` |
| 13 | 3.9 | 2.1 (của 7) | **không gần hơn → break** | `[7, 8, 10, 12]` |

Giá trị `15` không bao giờ được đọc.

In [20]:
def closest_k_window(root, target, k):
    window = deque()
    stack, node = [], root
    while stack or node is not None:
        while node is not None:
            stack.append(node)
            node = node.left
        node = stack.pop()
        x = node.val
        if len(window) < k:
            window.append(x)
        elif abs(x - target) < abs(window[0] - target):
            window.popleft()                 # trượt cửa sổ sang phải
            window.append(x)
        else:
            break                            # x và mọi giá trị sau đều xa hơn -> dừng sớm
        node = node.right
    return list(window)


def trace_window(root, target, k, verbose=True):
    window, stack, node, visited = deque(), [], root, 0
    while stack or node is not None:
        while node is not None:
            stack.append(node)
            node = node.left
        node = stack.pop()
        x = node.val
        visited += 1
        if len(window) < k:
            action = "chưa đủ k -> thêm"
            window.append(x)
        elif abs(x - target) < abs(window[0] - target):
            action = f"gần hơn {window[0]} -> bỏ {window[0]}, thêm {x}"
            window.popleft()
            window.append(x)
        else:
            action = f"không gần hơn {window[0]} -> BREAK"
        if verbose:
            print(f"x={x:>2} |x-t|={abs(x - target):.1f}  {action:<28} window={list(window)}")
        if action.endswith("BREAK"):
            break
        node = node.right
    return list(window), visited


res, visited = trace_window(root_T, 9.1, 4)
print("Kết quả:", res, f"| đã đọc {visited}/10 giá trị")
for tgt, kk in [(2.6, 2), (14.6, 4)]:
    res, visited = trace_window(root_T, tgt, kk, verbose=False)
    print(f"target={tgt}, k={kk}: kết quả {res}, đã đọc {visited}/10 giá trị")

x= 1 |x-t|=8.1  chưa đủ k -> thêm            window=[1]
x= 3 |x-t|=6.1  chưa đủ k -> thêm            window=[1, 3]
x= 4 |x-t|=5.1  chưa đủ k -> thêm            window=[1, 3, 4]
x= 6 |x-t|=3.1  chưa đủ k -> thêm            window=[1, 3, 4, 6]
x= 7 |x-t|=2.1  gần hơn 1 -> bỏ 1, thêm 7    window=[3, 4, 6, 7]
x= 8 |x-t|=1.1  gần hơn 3 -> bỏ 3, thêm 8    window=[4, 6, 7, 8]
x=10 |x-t|=0.9  gần hơn 4 -> bỏ 4, thêm 10   window=[6, 7, 8, 10]
x=12 |x-t|=2.9  gần hơn 6 -> bỏ 6, thêm 12   window=[7, 8, 10, 12]
x=13 |x-t|=3.9  không gần hơn 7 -> BREAK     window=[7, 8, 10, 12]
Kết quả: [7, 8, 10, 12] | đã đọc 9/10 giá trị
target=2.6, k=2: kết quả [3, 4], đã đọc 4/10 giá trị
target=14.6, k=4: kết quả [10, 12, 13, 15], đã đọc 10/10 giá trị


### 8.7 Độ phức tạp và nút thắt của cách 3b

- **Time**: số giá trị đọc được = (các giá trị đứng trước điểm dừng) + 1. Nếu `target` nhỏ (gần giá trị nhỏ nhất), dừng rất sớm (ví dụ `target = 2.6` chỉ đọc 4/10). Nhưng nếu `target` gần hoặc lớn hơn giá trị lớn nhất (ví dụ `14.6`), phải đọc **hết** cả 10 giá trị → **worst case `O(n)`**.
- **Space** `O(h + k)`: stack của phép duyệt sâu tối đa `h`, cửa sổ `k` phần tử. Không cần mảng `n` phần tử nữa.

Nút thắt cuối cùng: cách 3b **luôn xuất phát từ giá trị nhỏ nhất** rồi đi dần lên. Trong khi cách 3a chỉ cần `bisect_left` là **nhảy thẳng** tới chỗ `target` rơi vào. Câu hỏi follow-up chính là: **trên BST, làm sao "nhảy thẳng" tới `target` rồi đi ra hai phía, không cần mảng?**

## 9. Cách 4 (cách chính) — Hai stack predecessor / successor, `O(h + k)`

### 9.1 Mấu chốt: hai "con trỏ" trên cây

Nhớ lại cách 3a: hai con trỏ `lo` (đi lùi từ `pred`) và `hi` (đi tiến từ `succ`). Trên cây không có chỉ số, nhưng ta có thể tạo hai **iterator** (bộ duyệt từng bước):

- **iterator succ**: lần lượt cho ra các giá trị `>= target` theo thứ tự **tăng dần** — đây chính là in-order bằng vòng lặp của 8.5, chỉ khác là bắt đầu từ `target` chứ không từ giá trị nhỏ nhất;
- **iterator pred**: lần lượt cho ra các giá trị `< target` theo thứ tự **giảm dần** — in-order **ngược** (phải → node → trái), bắt đầu từ `target`.

**"Nhảy thẳng tới `target`"** = đi đường tìm kiếm từ root như mục 5.2, tốn `O(h)`:

- node `< target` → đẩy vào **stack `pred`**, rẽ **phải**;
- node `>= target` → đẩy vào **stack `succ`**, rẽ **trái**.

Đỉnh stack `pred` = node rẽ phải cuối cùng = **pred** của `target`; đỉnh stack `succ` = **succ** của `target`. Hai danh sách "nhóm `<`" và "nhóm `>=`" ở mục 5.2 chính là hai stack này (ở đây lưu **node** chứ không lưu giá trị, vì lát nữa cần đi tiếp xuống `.left` / `.right`).

In [21]:
def init_stacks(root, target):
    pred, succ = [], []              # pred: các node < target; succ: các node >= target
    node = root
    while node is not None:
        if node.val < target:
            pred.append(node)        # ứng viên phía trái; rẽ PHẢI tìm giá trị lớn hơn
            node = node.right
        else:
            succ.append(node)        # ứng viên phía phải; rẽ TRÁI tìm giá trị nhỏ hơn
            node = node.left
    return pred, succ


def vals(stack):
    return [n.val for n in stack]


for tgt in [9.1, 6, 0.5, 20]:
    p, s = init_stacks(root_T, tgt)
    print(f"target={tgt:<4} stack pred={str(vals(p)):<11} stack succ={str(vals(s)):<10} "
          f"đỉnh pred={p[-1].val if p else None}, đỉnh succ={s[-1].val if s else None}")

target=9.1  stack pred=[8]         stack succ=[13, 10]   đỉnh pred=8, đỉnh succ=10
target=6    stack pred=[3, 4]      stack succ=[8, 6]     đỉnh pred=4, đỉnh succ=6
target=0.5  stack pred=[]          stack succ=[8, 3, 1]  đỉnh pred=None, đỉnh succ=1
target=20   stack pred=[8, 13, 15] stack succ=[]         đỉnh pred=15, đỉnh succ=None


Nhận xét: từ đáy lên đỉnh, stack `pred` **tăng dần** (`[8, 13, 15]` với `target = 20`), còn stack `succ` **giảm dần** (`[8, 3, 1]` với `target = 0.5`). Đỉnh luôn là phần tử **gần `target` nhất** của mỗi phía. Với `target = 6` (trùng giá trị một node), `6` nằm ở đỉnh `succ` vì quy ước "bằng thì về phía `succ`".

### 9.2 Lấy giá trị kế tiếp của mỗi phía

**`next_succ`** (đi tiến — **y hệt** bước 2–4 + bước 1 của in-order vòng lặp ở 8.5):

```text
node = succ.pop()                     # giá trị nhỏ nhất >= target chưa dùng
cur = node.right
while cur: succ.append(cur); cur = cur.left     # đẩy "chuỗi trái" của cây con phải
return node.val
```

**`next_pred`** (đi lùi — đối xứng, đổi `left` ↔ `right`):

```text
node = pred.pop()                     # giá trị lớn nhất < target chưa dùng
cur = node.left
while cur: pred.append(cur); cur = cur.right    # đẩy "chuỗi phải" của cây con trái
return node.val
```

**Vì sao `next_pred` cho đúng giá trị nhỏ hơn kế tiếp?** Vừa lấy `p`. Các giá trị chưa dùng mà nhỏ hơn `p` và lớn nhất có thể nằm ở **cây con trái của `p`** (chúng nhỏ hơn `p`, nhưng lớn hơn node bên dưới `p` trong stack, vì `p` nằm trong cây con phải của node đó). Giá trị lớn nhất trong cây con trái của `p` là node **phải nhất** của nó → đi `p.left` rồi đi **phải hết cỡ**. Đẩy cả chuỗi vào stack để các node trên chuỗi được dùng dần về sau. Nếu `p.left` là `None` thì giá trị kế tiếp là node ngay dưới trong stack.

Ví dụ cây T, `target = 9.1`: lấy `8` xong, đi `8.left = 3`, rồi phải hết cỡ `3 → 6 → 7`; đẩy `[3, 6, 7]`, đỉnh mới là `7` — đúng là giá trị ngay dưới `8` trong dãy tăng dần.

Cell dưới "vắt cạn" hai iterator: phía `pred` phải ra dãy **giảm dần** các giá trị `< 9.1`, phía `succ` ra dãy **tăng dần** các giá trị `>= 9.1`.

In [22]:
def next_pred(pred):
    node = pred.pop()                # giá trị lớn nhất < target chưa dùng
    cur = node.left
    while cur is not None:           # đẩy chuỗi PHẢI của cây con trái
        pred.append(cur)
        cur = cur.right
    return node.val


def next_succ(succ):
    node = succ.pop()                # giá trị nhỏ nhất >= target chưa dùng
    cur = node.right
    while cur is not None:           # đẩy chuỗi TRÁI của cây con phải
        succ.append(cur)
        cur = cur.left
    return node.val


pred, succ = init_stacks(root_T, 9.1)
down = []
while pred:
    down.append(next_pred(pred))
up = []
while succ:
    up.append(next_succ(succ))
print("đi lùi từ target (pred):", down)
print("đi tiến từ target (succ):", up)
arr = inorder_list(root_T)
assert down == [x for x in reversed(arr) if x < 9.1]
assert up == [x for x in arr if x >= 9.1]
print("khớp với mảng in-order:", arr)

đi lùi từ target (pred): [8, 7, 6, 4, 3, 1]
đi tiến từ target (succ): [10, 12, 13, 15]
khớp với mảng in-order: [1, 3, 4, 6, 7, 8, 10, 12, 13, 15]


### 9.3 Ghép lại: "trộn từ giữa ra" như hai con trỏ

Giờ ta có đúng tình huống của cách 3a, chỉ thay "con trỏ trên mảng" bằng "stack trên cây". Lặp `k` lần:

- nếu stack `succ` rỗng → lấy từ `pred`;
- nếu stack `pred` rỗng → lấy từ `succ`;
- ngược lại so `target - pred[-1].val` với `succ[-1].val - target`; phía nào **gần hơn** thì lấy phía đó (gọi `next_pred` hoặc `next_succ`).

Điều kiện gộp gọn trong một dòng:

```python
if not succ or (pred and target - pred[-1].val <= succ[-1].val - target):
    lấy từ pred
else:
    lấy từ succ
```

**Về chuyện hòa (`<=`):** giả sử ở một bước hai đỉnh cách `target` bằng nhau, ta lấy `pred`. Bước sau, giá trị `pred` kế tiếp chắc chắn **xa hơn hẳn** (các giá trị khác nhau), nên đỉnh `succ` (vẫn ở khoảng cách cũ) sẽ được lấy ngay. Hai giá trị hòa vì thế luôn được lấy **liền nhau**. Chỉ khi chuyện hòa rơi đúng vào **lượt thứ `k`** (chỉ còn một chỗ cho hai ứng viên) thì kết quả mới phụ thuộc cách chọn — mà đó chính là trường hợp "đáp án không duy nhất" đề đã loại trừ.

In [23]:
def closest_k_two_stacks(root, target, k):
    pred, succ = init_stacks(root, target)                     # O(h)
    res = []
    for _ in range(k):
        if not succ or (pred and target - pred[-1].val <= succ[-1].val - target):
            res.append(next_pred(pred))                         # phía trái gần hơn
        else:
            res.append(next_succ(succ))                         # phía phải gần hơn
    return res


print(closest_k_two_stacks(build_tree([4, 2, 5, 1, 3]), 3.714286, 2))   # Example 1 -> [4, 3]
print(closest_k_two_stacks(build_tree([1]), 0.0, 1))                    # Example 2 -> [1]
print(closest_k_two_stacks(root_T, 9.1, 4))                             # cây T -> [10, 8, 7, 12]

[4, 3]
[1]
[10, 8, 7, 12]


## 10. Trace từng bước cách 4 trên cây T, `target = 9.1`, `k = 4`

```text
  ___8_____
 /         \
 3_     __13_
/  \   /     \
1  6  10_   15
  / \    \
  4 7   12
```

**Khởi tạo** (đường tìm kiếm): `8 < 9.1` → `pred = [8]`, rẽ phải tới `13`; `13 >= 9.1` → `succ = [13]`, rẽ trái tới `10`; `10 >= 9.1` → `succ = [13, 10]`, rẽ trái gặp `None` → dừng.

| Bước | Đỉnh pred (t − p) | Đỉnh succ (s − t) | Lấy | Đẩy thêm | pred sau | succ sau | res |
|:---:|:---:|:---:|:---:|---|---|---|---|
| 1 | 8 (1.1) | 10 (0.9) | **10** (succ) | chuỗi trái của `10.right`: `[12]` | `[8]` | `[13, 12]` | `[10]` |
| 2 | 8 (1.1) | 12 (2.9) | **8** (pred) | chuỗi phải của `8.left`: `[3, 6, 7]` | `[3, 6, 7]` | `[13, 12]` | `[10, 8]` |
| 3 | 7 (2.1) | 12 (2.9) | **7** (pred) | `7.left = None`: không đẩy | `[3, 6]` | `[13, 12]` | `[10, 8, 7]` |
| 4 | 6 (3.1) | 12 (2.9) | **12** (succ) | `12.right = None`: không đẩy | `[3, 6]` | `[13]` | `[10, 8, 7, 12]` |

Các cột cần để ý:

- Bước 2 là bước "đắt" nhất: lấy `8` xong phải đẩy cả chuỗi `3 → 6 → 7` để `7` (giá trị ngay dưới `8`) lên đỉnh.
- Các node `1`, `4`, `15` **không bao giờ bị đụng tới**. Trên cây lớn và cân bằng, phần không bị đụng tới chiếm gần hết cây — đó là lý do cách 4 nhanh hơn `O(n)`.

In [24]:
def trace_two_stacks(root, target, k):
    pred, succ = init_stacks(root, target)
    print(f"Khởi tạo: pred={vals(pred)}  succ={vals(succ)}")
    res = []
    for step in range(1, k + 1):
        p = f"{pred[-1].val} ({target - pred[-1].val:.1f})" if pred else "rỗng"
        s = f"{succ[-1].val} ({succ[-1].val - target:.1f})" if succ else "rỗng"
        if not succ or (pred and target - pred[-1].val <= succ[-1].val - target):
            side, x = "pred", next_pred(pred)
        else:
            side, x = "succ", next_succ(succ)
        res.append(x)
        print(f"Bước {step}: đỉnh pred={p:<9} đỉnh succ={s:<9} -> lấy {x:>2} ({side})"
              f"  pred={vals(pred)} succ={vals(succ)} res={res}")
    return res


_ = trace_two_stacks(root_T, 9.1, 4)

Khởi tạo: pred=[8]  succ=[13, 10]
Bước 1: đỉnh pred=8 (1.1)   đỉnh succ=10 (0.9)  -> lấy 10 (succ)  pred=[8] succ=[13, 12] res=[10]
Bước 2: đỉnh pred=8 (1.1)   đỉnh succ=12 (2.9)  -> lấy  8 (pred)  pred=[3, 6, 7] succ=[13, 12] res=[10, 8]
Bước 3: đỉnh pred=7 (2.1)   đỉnh succ=12 (2.9)  -> lấy  7 (pred)  pred=[3, 6] succ=[13, 12] res=[10, 8, 7]
Bước 4: đỉnh pred=6 (3.1)   đỉnh succ=12 (2.9)  -> lấy 12 (succ)  pred=[3, 6] succ=[13] res=[10, 8, 7, 12]


### 10.1 Trace thứ hai: một stack cạn sạch — `target = 14.6`, `k = 4`

**Khởi tạo:** `8 < 14.6` → `pred = [8]`, rẽ phải; `13 < 14.6` → `pred = [8, 13]`, rẽ phải; `15 >= 14.6` → `succ = [15]`, rẽ trái gặp `None`.

| Bước | Đỉnh pred (t − p) | Đỉnh succ (s − t) | Lấy | Đẩy thêm | pred sau | succ sau | res |
|:---:|:---:|:---:|:---:|---|---|---|---|
| 1 | 13 (1.6) | 15 (0.4) | **15** (succ) | `15.right = None` | `[8, 13]` | `[]` | `[15]` |
| 2 | 13 (1.6) | rỗng | **13** (pred) | chuỗi phải của `13.left`: `[10, 12]` | `[8, 10, 12]` | `[]` | `[15, 13]` |
| 3 | 12 (2.6) | rỗng | **12** (pred) | `12.left = None` | `[8, 10]` | `[]` | `[15, 13, 12]` |
| 4 | 10 (4.6) | rỗng | **10** (pred) | `10.left = None` | `[8]` | `[]` | `[15, 13, 12, 10]` |

Từ bước 2, `succ` rỗng → điều kiện `not succ` đúng → luôn lấy từ `pred`. Nếu quên kiểm tra rỗng, dòng `succ[-1]` sẽ ném `IndexError` (xem Lỗi 7 ở mục 15).

Cuối cell còn chạy thêm `target = 6` (trùng giá trị node), `k = 2`: `6` nằm ở đỉnh `succ` với khoảng cách `0.0` nên được lấy đầu tiên, rồi tới `7`.

In [25]:
_ = trace_two_stacks(root_T, 14.6, 4)
print()
_ = trace_two_stacks(root_T, 6, 2)

Khởi tạo: pred=[8, 13]  succ=[15]
Bước 1: đỉnh pred=13 (1.6)  đỉnh succ=15 (0.4)  -> lấy 15 (succ)  pred=[8, 13] succ=[] res=[15]
Bước 2: đỉnh pred=13 (1.6)  đỉnh succ=rỗng      -> lấy 13 (pred)  pred=[8, 10, 12] succ=[] res=[15, 13]
Bước 3: đỉnh pred=12 (2.6)  đỉnh succ=rỗng      -> lấy 12 (pred)  pred=[8, 10] succ=[] res=[15, 13, 12]
Bước 4: đỉnh pred=10 (4.6)  đỉnh succ=rỗng      -> lấy 10 (pred)  pred=[8] succ=[] res=[15, 13, 12, 10]

Khởi tạo: pred=[3, 4]  succ=[8, 6]
Bước 1: đỉnh pred=4 (2.0)   đỉnh succ=6 (0.0)   -> lấy  6 (succ)  pred=[3, 4] succ=[8, 7] res=[6]
Bước 2: đỉnh pred=4 (2.0)   đỉnh succ=7 (1.0)   -> lấy  7 (succ)  pred=[3, 4] succ=[8] res=[6, 7]


## 11. Code nộp LeetCode

Chuyển cách 4 vào đúng khung `class Solution`. Trên LeetCode **không** cần tự định nghĩa `TreeNode` và cũng không cần `import` `List`, `Optional` (môi trường đã có sẵn); trong notebook ta đã làm ở mục 2.

Vài điểm khi chuyển sang bản nộp:

- `next_pred` / `next_succ` viết thành **hàm lồng** bên trong `closestKValues`; chúng sửa `pred`, `succ` bằng `append`/`pop` (thay đổi **nội dung** list, không gán lại biến) nên **không cần `nonlocal`**.
- Toàn bộ là **vòng lặp**, không đệ quy → cây lệch `10^4` tầng vẫn chạy (bản đệ quy dễ dính `RecursionError`, xem Lỗi 6).
- `while node:` tương đương `while node is not None:` (node thật luôn là "truthy").

Ngay sau là `SolutionWindow` — bản nộp của **cách 3b** (đơn giản, `O(n)`), hữu ích khi phỏng vấn muốn đưa ra một lời giải đúng thật nhanh trước khi tối ưu.

Bộ test chọn để bắt các nhánh dễ sai:

| Test | Bắt lỗi gì |
|---|---|
| Example 1, Example 2 | đúng đề |
| `k = n` | lấy hết, cả hai stack đều phải cạn đúng lúc |
| `target = -10^9` (nhỏ hơn mọi giá trị) | stack `pred` rỗng ngay từ đầu |
| `target = 10^9` (lớn hơn mọi giá trị) | stack `succ` rỗng ngay từ đầu |
| `target` trùng giá trị một node | quy ước "bằng thì về `succ`", không lấy trùng |
| giá trị cỡ `10^9` | không có vấn đề tràn số / sai số với float |
| cây lệch `10^4` node | không đệ quy, `h = n` vẫn chạy |

Vì đề cho trả **thứ tự bất kỳ**, hàm `check` so sánh **sau khi sắp xếp** hai list.

In [26]:
class Solution:
    def closestKValues(self, root: Optional[TreeNode], target: float, k: int) -> List[int]:
        pred, succ = [], []                    # pred: node < target, succ: node >= target
        node = root
        while node:                            # đi đường tìm kiếm target: O(h)
            if node.val < target:
                pred.append(node)
                node = node.right
            else:
                succ.append(node)
                node = node.left

        def next_pred() -> int:                # giá trị kế tiếp đi lùi (giảm dần)
            node = pred.pop()
            cur = node.left
            while cur:
                pred.append(cur)
                cur = cur.right
            return node.val

        def next_succ() -> int:                # giá trị kế tiếp đi tiến (tăng dần)
            node = succ.pop()
            cur = node.right
            while cur:
                succ.append(cur)
                cur = cur.left
            return node.val

        res = []
        for _ in range(k):
            if not succ or (pred and target - pred[-1].val <= succ[-1].val - target):
                res.append(next_pred())
            else:
                res.append(next_succ())
        return res

In [27]:
class SolutionWindow:
    # Cách 3b: in-order bằng vòng lặp + cửa sổ trượt, dừng sớm. O(n) time, O(h + k) space.
    def closestKValues(self, root: Optional[TreeNode], target: float, k: int) -> List[int]:
        window = deque()
        stack, node = [], root
        while stack or node:
            while node:
                stack.append(node)
                node = node.left
            node = stack.pop()
            if len(window) < k:
                window.append(node.val)
            elif abs(node.val - target) < abs(window[0] - target):
                window.popleft()
                window.append(node.val)
            else:
                break
            node = node.right
        return list(window)


def check(name, actual, expected):
    assert sorted(actual) == sorted(expected), (name, actual, expected)
    shown = actual if len(actual) <= 6 else actual[:6] + ["..."]
    print(f"{name:<30} PASS {shown}")


chain = build_tree([v for i in range(10_000) for v in (i, None)][:-1])   # 0 -> 1 -> ... -> 9999 (lệch phải)

for sol in [Solution(), SolutionWindow()]:
    print("==", type(sol).__name__)
    check("Example 1", sol.closestKValues(build_tree([4, 2, 5, 1, 3]), 3.714286, 2), [4, 3])
    check("Example 2", sol.closestKValues(build_tree([1]), 0.0, 1), [1])
    check("k = n (lấy hết)", sol.closestKValues(build_tree([4, 2, 5, 1, 3]), 3.714286, 5), [1, 2, 3, 4, 5])
    check("target nhỏ hơn mọi giá trị", sol.closestKValues(root_T, -1e9, 3), [1, 3, 4])
    check("target lớn hơn mọi giá trị", sol.closestKValues(root_T, 1e9, 2), [13, 15])
    check("target trùng giá trị node", sol.closestKValues(root_T, 6, 2), [6, 7])
    check("cây T, target 9.1", sol.closestKValues(root_T, 9.1, 4), [7, 8, 10, 12])
    check("giá trị cỡ 10^9", sol.closestKValues(build_tree([10**9 - 1, 0, 10**9]), 999999999.6, 1), [10**9])
    check("cây lệch 10^4 node", sol.closestKValues(chain, 5000.3, 3), [4999, 5000, 5001])

== Solution
Example 1                      PASS [4, 3]
Example 2                      PASS [1]
k = n (lấy hết)                PASS [4, 3, 5, 2, 1]
target nhỏ hơn mọi giá trị     PASS [1, 3, 4]
target lớn hơn mọi giá trị     PASS [15, 13]
target trùng giá trị node      PASS [6, 7]
cây T, target 9.1              PASS [10, 8, 7, 12]
giá trị cỡ 10^9                PASS [1000000000]
cây lệch 10^4 node             PASS [5000, 5001, 4999]
== SolutionWindow
Example 1                      PASS [3, 4]
Example 2                      PASS [1]
k = n (lấy hết)                PASS [1, 2, 3, 4, 5]
target nhỏ hơn mọi giá trị     PASS [1, 3, 4]
target lớn hơn mọi giá trị     PASS [13, 15]
target trùng giá trị node      PASS [6, 7]
cây T, target 9.1              PASS [7, 8, 10, 12]
giá trị cỡ 10^9                PASS [1000000000]
cây lệch 10^4 node             PASS [4999, 5000, 5001]


## 12. Đối chiếu chéo các cách trên BST ngẫu nhiên

### 12.1 Sinh test mà không vướng chuyện hòa

Ví dụ tự nghĩ dễ bỏ sót trường hợp, nên ta sinh nhiều BST ngẫu nhiên nhỏ và so **tất cả các cách** với **brute force** (cách 1 — gần như "đúng theo định nghĩa").

Vấn đề: nếu input bị **hòa ở ranh giới** (ví dụ `target = 3.5`, `k = 1`, cây có cả `3` và `4`), đáp án không duy nhất, các cách khác nhau có thể trả về tập khác nhau mà **cách nào cũng đúng**. Đề loại trừ trường hợp này, nên test cũng phải loại trừ. Hai lớp phòng vệ:

1. Nhớ lại mục 1.2: `x ≠ y` hòa khoảng cách ⇔ `target = (x + y) / 2`. Giá trị node là **số nguyên** nên `(x + y) / 2` luôn là số nguyên hoặc "số nguyên + 0.5". Vậy nếu chọn `target` có **phần lẻ là 0.1, 0.25, 0.3, 0.7, 0.75 hoặc 0.9**, thì **không bao giờ hòa**. 70% số test dùng loại `target` này.
2. 30% còn lại cố ý dùng `target` nguyên hoặc ".5" (để thử nhánh "trùng giá trị node" và "hòa bên trong đáp án"). Với chúng, hàm `unique_answer` kiểm tra điều kiện `d_k < d_(k+1)` của mục 1.2; nếu vi phạm thì **bỏ qua** test đó — giống như LeetCode không bao giờ đưa input như vậy.

`random_bst(n)` lấy `n` số nguyên **khác nhau** rồi chèn theo thứ tự ngẫu nhiên (đủ kiểu hình dạng); 20% số lần chèn theo thứ tự tăng dần để tạo **cây lệch** (`h = n`). `random.seed(272)` cố định chuỗi ngẫu nhiên → chạy lại notebook luôn ra đúng các test đó. Kết quả so sánh **sau khi sắp xếp** (thứ tự bất kỳ).

In [28]:
def random_bst(n, lo=0, hi=60):
    vals_ = random.sample(range(lo, hi + 1), n)      # n giá trị đôi một khác nhau
    if random.random() < 0.2:
        vals_.sort()                                  # thỉnh thoảng tạo cây lệch (h = n)
    return build_bst(vals_)


def unique_answer(values, target, k):
    # True nếu tập k giá trị gần nhất là duy nhất (không hòa ở ranh giới thứ k / k+1).
    d = sorted(abs(x - target) for x in values)
    return k == len(d) or d[k - 1] < d[k]


methods = {
    "sort": closest_k_sort,
    "heap": closest_k_heap,
    "bisect": closest_k_bisect,
    "window": closest_k_window,
    "two_stacks": closest_k_two_stacks,
    "Solution": lambda r, t, kk: Solution().closestKValues(r, t, kk),
    "SolutionWindow": lambda r, t, kk: SolutionWindow().closestKValues(r, t, kk),
}

random.seed(272)
tested = skipped = 0
while tested < 1000:
    n = random.randint(1, 15)
    root = random_bst(n)
    k = random.randint(1, n)
    if random.random() < 0.7:
        target = random.randint(-5, 65) + random.choice([0.1, 0.25, 0.3, 0.7, 0.75, 0.9])
    else:
        target = random.randint(-5, 65) + random.choice([0, 0.5])    # có thể hòa -> phải kiểm tra
    values = inorder_list(root)
    if not unique_answer(values, target, k):
        skipped += 1
        continue
    expected = sorted(closest_k_sort(root, target, k))
    for name, f in methods.items():
        got = sorted(f(root, target, k))
        assert got == expected, (name, tree_to_list(root), target, k, got, expected)
    tested += 1

print(f"{tested} bộ test PASS cho cả {len(methods)} cách; bỏ qua {skipped} bộ bị hòa ở ranh giới")

random.seed(5)
demo = random_bst(8)
print("\nMột BST ngẫu nhiên:", tree_to_list(demo))
print_tree(demo)
print("target = 30.3, k = 3 ->", sorted(closest_k_two_stacks(demo, 30.3, 3)))

1000 bộ test PASS cho cả 7 cách; bỏ qua 8 bộ bị hòa ở ranh giới

Một BST ngẫu nhiên: [39, 16, 47, None, 22, 44, 50, None, None, None, None, None, 53, None, 58]
  __39___
 /       \
16_     47_
   \   /   \
  22  44  50_
             \
            53_
               \
              58
target = 30.3, k = 3 -> [22, 39, 44]


### 12.2 Bằng chứng cho follow-up: đếm số node bị "đụng tới"

Thời gian chạy đo bằng đồng hồ thì dao động theo máy; ta đếm thứ **chắc chắn**: **số node khác nhau bị đọc `.val`**. Mẹo: tạo `CountingTreeNode` kế thừa `TreeNode`, biến `val` thành một `@property` — mỗi lần ai đó đọc `node.val`, node đó được ghi vào tập `touched`. (Không cần hiểu sâu `@property`; chỉ cần biết: **đọc `.val` = bị ghi nhận**. Các hàm của ta không cần sửa dòng nào.)

Dựng một BST **cân bằng hoàn hảo** `n = 2^13 − 1 = 8191` node (`h = 13`, nằm trong giới hạn `n <= 10^4`), giá trị `0, 10, 20, ..., 81900`, `k = 5`, thử ba `target`: gần giá trị nhỏ nhất, ở giữa, gần giá trị lớn nhất.

In [29]:
class CountingTreeNode(TreeNode):
    touched = set()                          # id của các node đã bị đọc .val

    @property
    def val(self):
        CountingTreeNode.touched.add(id(self))
        return self._val

    @val.setter
    def val(self, v):
        self._val = v


n_big = 2 ** 13 - 1
big = build_balanced_bst([10 * i for i in range(n_big)], node_cls=CountingTreeNode)
print("n =", n_big, "| h =", height(big), "| k = 5")

funcs = [closest_k_sort, closest_k_heap, closest_k_bisect, closest_k_window, closest_k_two_stacks]
names = ["sort", "heap", "bisect", "window", "two_stacks"]
print(f"{'target':>9} | " + " | ".join(f"{nm:>10}" for nm in names))
for target in [3.3, 40_003.3, 81_803.3]:
    row, answers = [], set()
    for f in funcs:
        CountingTreeNode.touched.clear()
        answers.add(tuple(sorted(f(big, target, 5))))
        row.append(len(CountingTreeNode.touched))
    assert len(answers) == 1                 # mọi cách ra cùng một đáp án
    print(f"{target:>9} | " + " | ".join(f"{c:>10}" for c in row))

n = 8191 | h = 13 | k = 5
   target |       sort |       heap |     bisect |     window | two_stacks
      3.3 |       8191 |       8191 |       8191 |          6 |         15
  40003.3 |       8191 |       8191 |       8191 |       4004 |         15
  81803.3 |       8191 |       8191 |       8191 |       8184 |         15


**Đọc bảng ở mục 12.2:**

- `sort`, `heap`, `bisect` luôn đọc **cả 8191 node**, bất kể `target` ở đâu.
- `window` phụ thuộc vị trí `target`: gần giá trị nhỏ nhất thì chỉ đọc **6** node (dừng rất sớm), ở giữa đọc khoảng **một nửa** (4004), gần giá trị lớn nhất thì đọc gần **hết** (8184).
- `two_stacks` chỉ đọc **15** node ở cả ba vị trí: khoảng `h = 13` node trên đường tìm kiếm + vài node để lấy `k = 5` giá trị. Đó chính là `O(h + k)` — **nhỏ hơn `O(n)` rất nhiều** khi cây cân bằng. Câu follow-up đã được trả lời.

## 13. Vì sao cách 4 đúng?

Viết các bất biến (invariant) bằng lời — điều luôn đúng trước và sau mỗi bước:

**(I1) Stack `pred`.** Từ đáy lên đỉnh các node **tăng dần**, và tập các giá trị `< target` **chưa được lấy** đúng bằng: mỗi node `s` trong stack **cùng toàn bộ cây con trái của `s`**. Do đó **đỉnh stack = giá trị lớn nhất `< target` chưa được lấy**.

- *Sau khởi tạo:* mục 5.2 đã chứng minh mọi giá trị `v < target` hoặc nằm trên đường đi (đã ở trong stack), hoặc nằm trong cây con trái của một node rẽ phải (một node trong stack). Đúng (I1).
- *Sau `next_pred`:* lấy đỉnh `p` ra; phần "của `p`" còn lại là cây con trái của `p`. Ta đẩy chuỗi `c₁ = p.left, c₂ = c₁.right, c₃ = c₂.right, ...`. Cây con trái của `p` = `c₁` + cây con trái của `c₁` + cây con phải của `c₁`, mà cây con phải của `c₁` lại = `c₂` + cây con trái của `c₂` + ... Cứ thế, phần "của `p`" được chia đúng thành "mỗi `cᵢ` cùng cây con trái của nó". Mỗi `cᵢ` lớn hơn node bên dưới (vì nằm trong cây con phải của nó) → stack vẫn tăng dần. (I1) vẫn đúng.

**(I2) Stack `succ`** — đối xứng hoàn toàn: đỉnh = giá trị nhỏ nhất `>= target` chưa được lấy.

**(I3) Sau `j` lượt, `res` là `j` giá trị gần `target` nhất.** Giá trị chưa lấy gần nhất phải là **đỉnh `pred` hoặc đỉnh `succ`**: mọi giá trị chưa lấy khác ở phía trái đều **nhỏ hơn** đỉnh `pred` (nên xa hơn), ở phía phải đều **lớn hơn** đỉnh `succ` (nên xa hơn). Ta lấy đúng đỉnh gần hơn → quy nạp theo `j`, sau `k` lượt `res` là `k` giá trị gần nhất.

**Không lấy trùng, không bỏ sót:** mỗi node được đẩy vào **đúng một** stack (do `<` / `>=` chia tách) và bị `pop` **nhiều nhất một lần**.

## 14. Độ phức tạp và so sánh các cách

**Vì sao cách 4 là `O(h + k)`?** Khởi tạo đi một đường: `O(h)`. Sau đó, mỗi node được đẩy vào stack thì hoặc bị `pop` về sau (cả bài chỉ có `k` lần `pop`), hoặc còn nằm lại trong stack khi kết thúc (mỗi stack tại mọi thời điểm chỉ chứa các node trên **một** đường từ root xuống, nên `<= h` node). Tổng số lần đẩy `<= k + 2h` → tổng thời gian `O(h + k)`. Một lần `next_pred` riêng lẻ có thể tốn tới `O(h)` (như bước 2 ở mục 10 đẩy 3 node), nhưng **tính gộp** (amortized) thì vẫn là `O(h + k)`.

| Cách | Ý tưởng | Time | Bộ nhớ thêm | Dùng tính chất BST? | Khi nào dùng |
|---|---|:---:|:---:|:---:|---|
| 1. Sort | gom hết, sắp theo khoảng cách | `O(n log n)` | `O(n)` | Không | viết nhanh, làm "trọng tài" khi test |
| 2. Max-heap | giữ `k` ứng viên, đá phần tử xa nhất | `O(n log k)` | `O(k + h)` | Không | cây nhị phân **bất kỳ**, `k` nhỏ |
| 3a. Bisect + 2 con trỏ | in-order → mảng, mở rộng từ `target` | `O(n)` (+ `O(log n + k)` mỗi truy vấn sau khi có mảng) | `O(n)` | Có | **nhiều** truy vấn trên cùng một cây |
| 3b. Cửa sổ trượt | in-order + `deque`, dừng sớm | `O(n)` worst | `O(h + k)` | Có | lời giải `O(n)` gọn, ít bộ nhớ |
| **4. Hai stack** | hai iterator từ `target` ra hai phía | **`O(h + k)`** | `O(h)` (+ `O(k)` kết quả) | Có | **cách chính**, follow-up |

Ghi chú:

- Với BST **cân bằng**, `h = O(log n)` → cách 4 là `O(log n + k)`, nhỏ hơn `O(n)` khi `k` nhỏ so với `n`. Nếu `k` gần bằng `n` thì chỉ riêng việc **in ra** `k` giá trị đã tốn `O(n)` — không thuật toán nào nhanh hơn được.
- Với BST **lệch** (`h = n`), cách 4 cũng thành `O(n)`. Đó là lý do follow-up phải nói "giả sử BST cân bằng".
- `n <= 10^4` nên trên LeetCode **cả 5 cách đều qua**; điểm khác biệt nằm ở tư duy và câu follow-up khi phỏng vấn.

## 15. Lỗi thường gặp

### Lỗi 1: Khởi tạo hai stack bằng **hai lần đi riêng** với `<=` và `>=` → giá trị bằng `target` bị lấy hai lần

```python
# SAI: lần 1 gom node <= target, lần 2 gom node >= target
if node.val <= target: pred.append(node) ...
if node.val >= target: succ.append(node) ...
```

Khi `target` trùng giá trị một node (ví dụ `target = 6`), node `6` nằm ở **cả hai** stack → kết quả `[6, 6]`. **Sửa:** chia bằng **một** phép so sánh `node.val < target` → `pred`, còn lại → `succ`, trong **một** lần đi. Mỗi node thuộc đúng một phía.

### Lỗi 2: Sau khi `pop` ở `pred`, chỉ đẩy `node.left` mà quên **đi phải hết cỡ**

```python
node = pred.pop()
if node.left: pred.append(node.left)     # SAI: thiếu vòng while đi sang phải
```

Giá trị ngay dưới `node` là node **phải nhất** của cây con trái, không phải bản thân `node.left`. Trên cây T, `target = 9.1`: lấy `8` xong, bản sai đưa `3` lên đỉnh thay vì `7` → kết quả sai `[10, 8, 12, 13]` thay vì `[10, 8, 7, 12]`. **Sửa:** `cur = node.left; while cur: pred.append(cur); cur = cur.right`.

Cell dưới dùng hàm `merge_k` (phần "trộn" của cách 4, nhận sẵn hai stack) để cắm từng bản sai vào cho dễ so.

In [30]:
def merge_k(pred, succ, target, k, np=next_pred, ns=next_succ):
    res = []
    for _ in range(k):
        if not succ or (pred and target - pred[-1].val <= succ[-1].val - target):
            res.append(np(pred))
        else:
            res.append(ns(succ))
    return res


# ---- Lỗi 1
def init_stacks_two_walks_buggy(root, target):
    pred, succ = [], []
    node = root
    while node is not None:                  # lần đi 1: gom node <= target
        if node.val <= target:
            pred.append(node)
            node = node.right
        else:
            node = node.left
    node = root
    while node is not None:                  # lần đi 2: gom node >= target
        if node.val >= target:
            succ.append(node)
            node = node.left
        else:
            node = node.right
    return pred, succ


p, s = init_stacks_two_walks_buggy(root_T, 6)
print("Lỗi 1 - stack:", vals(p), vals(s), "-> 6 nằm ở cả hai")
print("Lỗi 1 - SAI :", merge_k(p, s, 6, 2))
print("Lỗi 1 - ĐÚNG:", merge_k(*init_stacks(root_T, 6), 6, 2))


# ---- Lỗi 2
def next_pred_buggy(pred):
    node = pred.pop()
    if node.left is not None:
        pred.append(node.left)               # SAI: chỉ đẩy con trái, quên đi phải hết cỡ
    return node.val


print("\nLỗi 2 - SAI :", merge_k(*init_stacks(root_T, 9.1), 9.1, 4, np=next_pred_buggy))
print("Lỗi 2 - ĐÚNG:", merge_k(*init_stacks(root_T, 9.1), 9.1, 4))

Lỗi 1 - stack: [3, 6] [8, 6] -> 6 nằm ở cả hai
Lỗi 1 - SAI : [6, 6]
Lỗi 1 - ĐÚNG: [6, 7]

Lỗi 2 - SAI : [10, 8, 12, 13]
Lỗi 2 - ĐÚNG: [10, 8, 7, 12]


### Lỗi 3: Heap quên **đổi dấu** khoảng cách

`heappush(heap, (abs(x - target), x))` tạo **min-heap** theo khoảng cách; khi đủ `k + 1` phần tử, `heappop` đá phần tử **gần nhất** ra → cuối cùng giữ lại `k` phần tử **xa nhất**! **Sửa:** đẩy `(-abs(x - target), x)`.

### Lỗi 4: Cửa sổ trượt so với `window[-1]` thay vì `window[0]`

Phần tử bị thay thế phải là phần tử **trái nhất** (`window[0]`). So với phần tử mới nhất `window[-1]` thì cửa sổ dừng quá sớm. Trên cây T, `target = 9.1`, `k = 4`: tới `x = 12`, bản sai so `|12 − 9.1| = 2.9` với `|10 − 9.1| = 0.9` → `break` → ra `[6, 7, 8, 10]`, trong khi đúng phải là `[7, 8, 10, 12]` (vì `12` gần hơn `6`).

### Lỗi 5: Sắp xếp theo `x - target` mà quên `abs`

`key=lambda x: x - target` xếp theo **hiệu có dấu** → các giá trị **nhỏ nhất** lên đầu (hiệu âm nhất), chứ không phải gần nhất. **Sửa:** `key=lambda x: abs(x - target)`.

### Lỗi 6: Duyệt đệ quy trên cây lệch `10^4` tầng → `RecursionError`

Python giới hạn độ sâu đệ quy (`sys.getrecursionlimit()`, thường là 1000 hoặc 3000 tùy môi trường). Cây lệch `10^4` node (hợp lệ theo constraints) làm `inorder` đệ quy vượt giới hạn. **Sửa:** dùng bản vòng lặp (cách 3b, cách 4 đều thuần vòng lặp), hoặc tăng giới hạn bằng `sys.setrecursionlimit` (phụ thuộc môi trường, kém an toàn hơn).

### Lỗi 7: Quên kiểm tra stack rỗng trước khi đọc `stack[-1]`

Khi `target` nằm gần một đầu của dãy giá trị (như `14.6` ở mục 10.1), một stack sẽ cạn trước. Đọc `succ[-1]` lúc `succ == []` → `IndexError`. **Sửa:** điều kiện `if not succ or (pred and ...)`.

In [31]:
# ---- Lỗi 3: quên đổi dấu
def closest_k_heap_wrong_sign(root, target, k):
    heap = []
    for x in inorder_list(root):
        heapq.heappush(heap, (abs(x - target), x))       # SAI: min-heap theo khoảng cách
        if len(heap) > k:
            heapq.heappop(heap)                          # -> đá phần tử GẦN nhất ra!
    return [x for _, x in heap]


print("Lỗi 3 - SAI :", sorted(closest_k_heap_wrong_sign(root_T, 9.1, 3)), " ĐÚNG:", sorted(closest_k_heap(root_T, 9.1, 3)))


# ---- Lỗi 4: so với window[-1]
def closest_k_window_wrong(root, target, k):
    window = deque()
    for x in inorder_list(root):
        if len(window) < k:
            window.append(x)
        elif abs(x - target) < abs(window[-1] - target):  # SAI: phải là window[0]
            window.popleft()
            window.append(x)
        else:
            break
    return list(window)


print("Lỗi 4 - SAI :", closest_k_window_wrong(root_T, 9.1, 4), " ĐÚNG:", closest_k_window(root_T, 9.1, 4))

# ---- Lỗi 5: quên abs
wrong5 = sorted(inorder_list(root_T), key=lambda x: x - 9.1)[:4]
print("Lỗi 5 - SAI :", wrong5, " ĐÚNG:", sorted(closest_k_sort(root_T, 9.1, 4)))

# ---- Lỗi 6: đệ quy trên cây lệch 10^4 tầng (biến chain dựng ở mục 11)
print("\nGiới hạn đệ quy hiện tại:", sys.getrecursionlimit())
try:
    closest_k_sort(chain, 5000.3, 3)                     # inorder đệ quy -> sâu 10^4 tầng
except RecursionError as e:
    print("Lỗi 6 - cách 1 (đệ quy):", type(e).__name__, "-", e)
print("Lỗi 6 - Solution (vòng lặp):", Solution().closestKValues(chain, 5000.3, 3))


# ---- Lỗi 7: không kiểm tra stack rỗng
def merge_k_no_empty_check(pred, succ, target, k):
    res = []
    for _ in range(k):
        if target - pred[-1].val <= succ[-1].val - target:    # SAI: stack có thể rỗng
            res.append(next_pred(pred))
        else:
            res.append(next_succ(succ))
    return res


try:
    merge_k_no_empty_check(*init_stacks(root_T, 14.6), 14.6, 4)
except IndexError as e:
    print("\nLỗi 7 -", type(e).__name__, "-", e)
print("Lỗi 7 - ĐÚNG:", merge_k(*init_stacks(root_T, 14.6), 14.6, 4))

Lỗi 3 - SAI : [1, 3, 15]  ĐÚNG: [7, 8, 10]
Lỗi 4 - SAI : [6, 7, 8, 10]  ĐÚNG: [7, 8, 10, 12]
Lỗi 5 - SAI : [1, 3, 4, 6]  ĐÚNG: [7, 8, 10, 12]

Giới hạn đệ quy hiện tại: 1000
Lỗi 6 - cách 1 (đệ quy): RecursionError - maximum recursion depth exceeded
Lỗi 6 - Solution (vòng lặp): [5000, 5001, 4999]

Lỗi 7 - IndexError - list index out of range
Lỗi 7 - ĐÚNG: [15, 13, 12, 10]


## 16. Mẹo nhận dạng pattern và bài liên quan

### Dấu hiệu nhận dạng

| Thấy trong đề | Nghĩ ngay tới |
|---|---|
| "BST" + "gần nhất / nhỏ thứ k / kế tiếp / floor / ceiling" | in-order **tăng dần**; đường tìm kiếm `O(h)` cho pred/succ |
| "k phần tử tốt nhất theo một tiêu chí", dữ liệu **không** có thứ tự | **heap kích thước k** (đá phần tử tệ nhất) |
| "k phần tử gần x nhất" trên **mảng đã sắp xếp** | đáp án là **đoạn liền** → bisect + hai con trỏ / cửa sổ |
| cần duyệt BST **từng bước**, **dừng giữa chừng**, hoặc duyệt **ngược** | **iterator in-order bằng stack** (đẩy chuỗi trái / chuỗi phải) |
| "đi từ một điểm ra hai phía", "hai đầu tiến vào giữa" | **hai con trỏ** — trên cây thì thành **hai stack** |
| follow-up "nhanh hơn `O(n)` trên BST cân bằng" | chỉ được đụng tới `O(h)` node + phần cần trả về |

### Bài liên quan

| Bài | Liên hệ |
|---|---|
| 270. Closest Binary Search Tree Value (Easy, Premium) | bản `k = 1`: chỉ cần pred/succ trên đường tìm kiếm (mục 5) |
| 658. Find K Closest Elements (Medium) | bản **mảng đã sắp xếp** của cách 3a (Bài tập 4) |
| 173. Binary Search Tree Iterator (Medium) | chính là iterator `succ` (đẩy chuỗi trái), xuất phát từ giá trị nhỏ nhất |
| 230. Kth Smallest Element in a BST (Medium) | in-order vòng lặp, dừng ở phần tử thứ `k` |
| 285. Inorder Successor in BST (Medium, Premium) | tìm `succ` của một node bằng đường tìm kiếm |
| 653. Two Sum IV - Input is a BST (Easy) | **hai iterator** tăng/giảm như hai con trỏ (Bài tập 6) |
| 98. Validate Binary Search Tree (Medium) | in-order phải tăng ngặt (mục 4.2) |
| 1305. All Elements in Two Binary Search Trees (Medium) | trộn **hai iterator** in-order của hai cây |

Cell dưới cài bài **173** theo đúng format LeetCode (tên class/method của đề) và chạy chuỗi lệnh mẫu của đề bằng hàm `run_commands`: bạn sẽ thấy nó **chính là** `next_succ` của cách 4, chỉ khác điểm xuất phát.

In [32]:
class BSTIterator:
    def __init__(self, root: Optional[TreeNode]):
        self.stack = []
        self._push_left(root)                # xuất phát từ giá trị nhỏ nhất

    def _push_left(self, node):
        while node:
            self.stack.append(node)
            node = node.left

    def next(self) -> int:
        node = self.stack.pop()
        self._push_left(node.right)          # pop rồi đẩy chuỗi trái của cây con phải
        return node.val

    def hasNext(self) -> bool:
        return len(self.stack) > 0


def run_commands(cls, commands, args):
    # Chạy chuỗi lệnh kiểu LeetCode: lệnh đầu là tên class (khởi tạo), các lệnh sau là method.
    out, obj = [], None
    for cmd, a in zip(commands, args):
        if cmd == cls.__name__:
            obj = cls(*a)
            out.append(None)
        else:
            out.append(getattr(obj, cmd)(*a))
    return out


cmds = ["BSTIterator", "next", "next", "hasNext", "next", "hasNext", "next", "hasNext", "next", "hasNext"]
args = [[build_tree([7, 3, 15, None, None, 9, 20])], [], [], [], [], [], [], [], [], []]
result = run_commands(BSTIterator, cmds, args)
print(result)
assert result == [None, 3, 7, True, 9, True, 15, True, 20, False]
print("#173 khớp output mẫu của đề")

[None, 3, 7, True, 9, True, 15, True, 20, False]
#173 khớp output mẫu của đề


## 17. Bài tập vận dụng

Làm theo thứ tự, **không** kéo xuống xem đáp án trước. Gợi ý theo mức ở mục 18, đáp án ở mục 19.

### Bài 1 — Đoán output (không chạy code)

Với cây T và `target = 4.4`, sau khi gọi `init_stacks(root_T, 4.4)`:

- `vals(pred)` và `vals(succ)` là gì (liệt kê từ đáy lên đỉnh)?
- Đỉnh của mỗi stack là giá trị nào? Chúng có đúng là pred / succ của `4.4` không?

### Bài 2 — Tính tay bằng bảng

Tiếp Bài 1, lập bảng trace cách 4 với `target = 4.4`, `k = 3` theo đúng các cột của mục 10: *Bước | Đỉnh pred (t − p) | Đỉnh succ (s − t) | Lấy | Đẩy thêm | pred sau | succ sau | res*. Kết quả cuối cùng là gì?

### Bài 3 — Điền TODO: tự viết lại cách 4

Hoàn thành `closest_k_exercise` dưới đây theo 4 TODO. Viết xong mới bỏ comment các dòng `assert` để tự kiểm tra.

In [33]:
def closest_k_exercise(root, target, k):
    pred, succ = [], []
    node = root
    # TODO 1: đi đường tìm kiếm target:
    #         node.val < target -> đẩy vào pred, rẽ phải; ngược lại -> đẩy vào succ, rẽ trái

    def take_pred():
        # TODO 2: pop đỉnh pred; đẩy chuỗi PHẢI của cây con trái; trả về giá trị vừa pop
        pass

    def take_succ():
        # TODO 3: pop đỉnh succ; đẩy chuỗi TRÁI của cây con phải; trả về giá trị vừa pop
        pass

    res = []
    # TODO 4: lặp k lần, lấy từ phía gần target hơn (nhớ trường hợp một stack rỗng)
    return res


# Tự làm xong rồi mới mở test:
# assert sorted(closest_k_exercise(build_tree([4, 2, 5, 1, 3]), 3.714286, 2)) == [3, 4]
# assert sorted(closest_k_exercise(root_T, 9.1, 4)) == [7, 8, 10, 12]
# assert sorted(closest_k_exercise(root_T, 14.6, 4)) == [10, 12, 13, 15]
# assert sorted(closest_k_exercise(root_T, -5, 2)) == [1, 3]

### Bài 4 — Biến thể: LeetCode 658. Find K Closest Elements

> Cho mảng số nguyên `arr` **đã sắp xếp tăng dần**, hai số nguyên `k` và `x`. Trả về `k` số trong mảng gần `x` nhất; kết quả cũng phải **sắp xếp tăng dần**.
>
> Số `a` được coi là gần `x` hơn số `b` nếu `|a - x| < |b - x|`, **hoặc** `|a - x| == |b - x|` và `a < b`.
>
> Ví dụ: `arr = [1,2,3,4,5], k = 4, x = 3` → `[1,2,3,4]`; `arr = [1,1,2,3,4,5], k = 4, x = -1` → `[1,1,2,3]`.
>
> Constraints: `1 <= k <= arr.length`, `1 <= arr.length <= 10^4`, `arr` tăng dần, `-10^4 <= arr[i], x <= 10^4`.
>
> Signature: `def findClosestElements(self, arr: List[int], k: int, x: int) -> List[int]:`

Khác bài 272 ở hai điểm: mảng **có thể có số trùng**, và **có quy tắc phá hòa** (số nhỏ hơn thắng) thay vì cam kết đáp án duy nhất. Hãy viết `find_closest_elements` bằng **bisect + hai con trỏ** (mục 8.3) và nghĩ xem điều kiện so sánh cần `<` hay `<=` để đúng quy tắc phá hòa.

In [34]:
def find_closest_elements(arr, k, x):
    # TODO: hi = bisect_left(arr, x), lo = hi - 1; mở rộng k lần (hòa -> ưu tiên bên nào?)
    #       rồi trả về đoạn arr[lo + 1:hi] (đã tăng dần sẵn).
    raise NotImplementedError


# assert find_closest_elements([1, 2, 3, 4, 5], 4, 3) == [1, 2, 3, 4]
# assert find_closest_elements([1, 1, 2, 3, 4, 5], 4, -1) == [1, 1, 2, 3]
# assert find_closest_elements([1, 3], 1, 2) == [1]            # hòa |1-2| = |3-2| -> số nhỏ hơn thắng

### Bài 5 — Biến thể: trả kết quả **tăng dần** mà không cần sắp xếp lại

Đề 272 cho thứ tự bất kỳ, nhưng giả sử người phỏng vấn đòi kết quả **tăng dần**. Gọi `sorted(res)` thì tốn thêm `O(k log k)`. Hãy viết `closest_k_sorted_output` vẫn trong `O(h + k)`: dùng một `deque` tên `out`, giá trị lấy từ `pred` thì đặt vào **bên trái** (`appendleft`), lấy từ `succ` thì đặt vào **bên phải** (`append`). Tự giải thích vì sao `out` luôn tăng dần. (Được dùng lại `init_stacks`, `next_pred`, `next_succ`.)

In [35]:
def closest_k_sorted_output(root, target, k):
    # TODO: giống closest_k_two_stacks, nhưng pred -> out.appendleft(...), succ -> out.append(...)
    raise NotImplementedError


# assert closest_k_sorted_output(root_T, 9.1, 4) == [7, 8, 10, 12]
# assert closest_k_sorted_output(root_T, 14.6, 4) == [10, 12, 13, 15]
# assert closest_k_sorted_output(build_tree([4, 2, 5, 1, 3]), 3.714286, 2) == [3, 4]

### Bài 6 — Mini challenge: LeetCode 653. Two Sum IV - Input is a BST

> Cho `root` của một BST và số nguyên `k`, trả về `True` nếu tồn tại **hai phần tử khác nhau** trong BST có tổng bằng `k`, ngược lại `False`.
>
> Ví dụ: `root = [5,3,6,2,4,null,7]`, `k = 9` → `True` (`2 + 7` hoặc `3 + 6` hoặc `4 + 5`); cùng cây, `k = 28` → `False`.
>
> Constraints: số node trong `[1, 10^4]`, `-10^4 <= Node.val <= 10^4`, `root` là BST hợp lệ, `-10^5 <= k <= 10^5`.
>
> Signature: `def findTarget(self, root: Optional[TreeNode], k: int) -> bool:`

Yêu cầu thêm: **không** dựng mảng in-order; chỉ dùng hai stack (bộ nhớ `O(h)`). Gợi ý tư duy: trên mảng tăng dần, Two Sum giải bằng hai con trỏ `i` (đầu) và `j` (cuối). Trên BST, con trỏ `i` = iterator **tăng dần** xuất phát từ giá trị nhỏ nhất, con trỏ `j` = iterator **giảm dần** xuất phát từ giá trị lớn nhất.

In [36]:
def two_sum_bst(root, k):
    # TODO: lo_stack = chuỗi TRÁI từ root (iterator tăng dần), hi_stack = chuỗi PHẢI từ root (giảm dần).
    #       Lặp khi giá trị bên trái < giá trị bên phải:
    #         tổng == k -> True; tổng < k -> tiến con trỏ trái; tổng > k -> lùi con trỏ phải.
    raise NotImplementedError


# assert two_sum_bst(build_tree([5, 3, 6, 2, 4, None, 7]), 9) is True
# assert two_sum_bst(build_tree([5, 3, 6, 2, 4, None, 7]), 28) is False
# assert two_sum_bst(build_tree([1]), 2) is False                 # không được dùng một phần tử hai lần

## 18. Gợi ý theo mức

Đọc **mức 1** trước; vẫn bí mới đọc mức 2, rồi mức 3.

**Bài 1**
- Mức 1: quy tắc khởi tạo — `node.val < target` → `pred` và rẽ **phải**; ngược lại → `succ` và rẽ **trái**.
- Mức 2: bắt đầu ở root `8`: `8 >= 4.4` nên `8` vào `succ`, rẽ trái tới `3`...
- Mức 3: đường đi là `8 → 3 → 6 → 4 → None`.

**Bài 2**
- Mức 1: mỗi bước chỉ so **hai đỉnh**; phía nào gần hơn thì lấy phía đó.
- Mức 2: lấy từ `pred` thì đẩy **chuỗi phải của cây con trái**; lấy từ `succ` thì đẩy **chuỗi trái của cây con phải**.
- Mức 3: bước 1 so `4` (0.4) với `6` (1.6); bước 2 lấy `3` xong phải đẩy node `1`.

**Bài 3**
- Mức 1: chép lại khung của `closest_k_two_stacks` + `init_stacks` + `next_pred` / `next_succ` (mục 9), nhưng gom vào một hàm.
- Mức 2: TODO 2 là `node = pred.pop(); cur = node.left; while cur: pred.append(cur); cur = cur.right; return node.val`. Hàm lồng không cần `nonlocal` vì chỉ gọi `append`/`pop` chứ không gán lại `pred`.
- Mức 3: TODO 4 dùng điều kiện `if not succ or (pred and target - pred[-1].val <= succ[-1].val - target)`.

**Bài 4**
- Mức 1: đáp án là một **đoạn liền** (mục 8.1) — vẫn đúng khi có số trùng.
- Mức 2: khi hòa, số **nhỏ hơn** thắng; số nhỏ hơn nằm ở **bên trái** → điều kiện nhận bên trái phải là `x - arr[lo] <= arr[hi] - x` (có dấu `=`).
- Mức 3: kết quả là `arr[lo + 1:hi]`, đã tăng dần sẵn, không cần sort.

**Bài 5**
- Mức 1: các giá trị lấy từ `pred` ra theo thứ tự **giảm dần**, mỗi giá trị mới **nhỏ hơn** mọi thứ đã có trong `out`.
- Mức 2: `out = deque()`; `out.appendleft(next_pred(pred))` / `out.append(next_succ(succ))`.
- Mức 3: cuối hàm `return list(out)`.

**Bài 6**
- Mức 1: hai con trỏ trên mảng tăng dần: tổng nhỏ quá thì tăng con trỏ trái, lớn quá thì giảm con trỏ phải; dừng khi hai con trỏ gặp nhau.
- Mức 2: iterator tăng dần = `next_succ` (đẩy chuỗi trái của cây con phải); iterator giảm dần = `next_pred` (đẩy chuỗi phải của cây con trái). Khởi tạo: đẩy chuỗi trái / chuỗi phải bắt đầu từ root.
- Mức 3: trong vòng lặp, đọc `a = lo_stack[-1]`, `b = hi_stack[-1]`; nếu `a is b` hoặc `a.val >= b.val` thì trả `False`.

## 19. Đáp án tham khảo

### Đáp án Bài 1

Đường đi với `target = 4.4`: `8 >= 4.4` → `succ = [8]`, rẽ trái; `3 < 4.4` → `pred = [3]`, rẽ phải; `6 >= 4.4` → `succ = [8, 6]`, rẽ trái; `4 < 4.4` → `pred = [3, 4]`, rẽ phải gặp `None`.

→ `vals(pred) = [3, 4]` (đỉnh `4`), `vals(succ) = [8, 6]` (đỉnh `6`). Đúng là pred (`4`, lớn nhất `< 4.4`) và succ (`6`, nhỏ nhất `>= 4.4`) — trong dãy `[1, 3, 4, 6, 7, ...]`, `4.4` rơi vào giữa `4` và `6`.

### Đáp án Bài 2

| Bước | Đỉnh pred (t − p) | Đỉnh succ (s − t) | Lấy | Đẩy thêm | pred sau | succ sau | res |
|:---:|:---:|:---:|:---:|---|---|---|---|
| 1 | 4 (0.4) | 6 (1.6) | **4** (pred) | `4.left = None` | `[3]` | `[8, 6]` | `[4]` |
| 2 | 3 (1.4) | 6 (1.6) | **3** (pred) | chuỗi phải của `3.left`: `[1]` | `[1]` | `[8, 6]` | `[4, 3]` |
| 3 | 1 (3.4) | 6 (1.6) | **6** (succ) | chuỗi trái của `6.right`: `[7]` | `[1]` | `[8, 7]` | `[4, 3, 6]` |

Kết quả `[4, 3, 6]` (tức tập `{3, 4, 6}`). Kiểm tra tính duy nhất: khoảng cách sắp tăng dần `0.4, 1.4, 1.6, 2.6 (của 7), ...` → `d_3 = 1.6 < d_4 = 2.6`, không hòa ở ranh giới.

### Đáp án Bài 3–6

Cell dưới kiểm tra lại Bài 1–2 bằng code, rồi cài đáp án Bài 3–6 với `assert` (kể cả đối chiếu ngẫu nhiên với brute force).

In [37]:
# ---- Kiểm tra Bài 1 & 2
p, s = init_stacks(root_T, 4.4)
assert vals(p) == [3, 4] and vals(s) == [8, 6]
print("Bài 1:", vals(p), vals(s))
assert trace_two_stacks(root_T, 4.4, 3) == [4, 3, 6]


# ---- Bài 3
def answer_closest_k_exercise(root, target, k):
    pred, succ = [], []
    node = root
    while node:                                          # TODO 1
        if node.val < target:
            pred.append(node)
            node = node.right
        else:
            succ.append(node)
            node = node.left

    def take_pred():                                     # TODO 2
        node = pred.pop()
        cur = node.left
        while cur:
            pred.append(cur)
            cur = cur.right
        return node.val

    def take_succ():                                     # TODO 3
        node = succ.pop()
        cur = node.right
        while cur:
            succ.append(cur)
            cur = cur.left
        return node.val

    res = []
    for _ in range(k):                                   # TODO 4
        if not succ or (pred and target - pred[-1].val <= succ[-1].val - target):
            res.append(take_pred())
        else:
            res.append(take_succ())
    return res


# ---- Bài 4 (LeetCode 658)
def answer_find_closest_elements(arr, k, x):
    hi = bisect_left(arr, x)
    lo = hi - 1
    for _ in range(k):
        if lo < 0:
            hi += 1
        elif hi >= len(arr):
            lo -= 1
        elif x - arr[lo] <= arr[hi] - x:                 # hòa -> số nhỏ hơn (bên trái) thắng
            lo -= 1
        else:
            hi += 1
    return arr[lo + 1:hi]


# ---- Bài 5
def answer_closest_k_sorted_output(root, target, k):
    pred, succ = init_stacks(root, target)
    out = deque()
    for _ in range(k):
        if not succ or (pred and target - pred[-1].val <= succ[-1].val - target):
            out.appendleft(next_pred(pred))              # nhỏ hơn mọi thứ đang có -> đặt bên trái
        else:
            out.append(next_succ(succ))                  # lớn hơn mọi thứ đang có -> đặt bên phải
    return list(out)


# ---- Bài 6 (LeetCode 653)
def answer_two_sum_bst(root, k):
    lo_stack, hi_stack = [], []
    node = root
    while node:                                          # iterator tăng dần: chuỗi trái từ root
        lo_stack.append(node)
        node = node.left
    node = root
    while node:                                          # iterator giảm dần: chuỗi phải từ root
        hi_stack.append(node)
        node = node.right
    while lo_stack and hi_stack:
        a, b = lo_stack[-1], hi_stack[-1]
        if a is b or a.val >= b.val:                     # hai con trỏ gặp nhau -> hết cặp để thử
            return False
        total = a.val + b.val
        if total == k:
            return True
        if total < k:
            next_succ(lo_stack)                          # tiến con trỏ trái
        else:
            next_pred(hi_stack)                          # lùi con trỏ phải
    return False


assert sorted(answer_closest_k_exercise(build_tree([4, 2, 5, 1, 3]), 3.714286, 2)) == [3, 4]
assert sorted(answer_closest_k_exercise(root_T, 9.1, 4)) == [7, 8, 10, 12]
assert sorted(answer_closest_k_exercise(root_T, 14.6, 4)) == [10, 12, 13, 15]
assert sorted(answer_closest_k_exercise(root_T, -5, 2)) == [1, 3]

assert answer_find_closest_elements([1, 2, 3, 4, 5], 4, 3) == [1, 2, 3, 4]
assert answer_find_closest_elements([1, 1, 2, 3, 4, 5], 4, -1) == [1, 1, 2, 3]
assert answer_find_closest_elements([1, 3], 1, 2) == [1]

assert answer_closest_k_sorted_output(root_T, 9.1, 4) == [7, 8, 10, 12]
assert answer_closest_k_sorted_output(root_T, 14.6, 4) == [10, 12, 13, 15]
assert answer_closest_k_sorted_output(build_tree([4, 2, 5, 1, 3]), 3.714286, 2) == [3, 4]

ex653 = build_tree([5, 3, 6, 2, 4, None, 7])
assert answer_two_sum_bst(ex653, 9) is True
assert answer_two_sum_bst(ex653, 28) is False
assert answer_two_sum_bst(build_tree([1]), 2) is False

# Đối chiếu ngẫu nhiên với brute force
random.seed(658)
for _ in range(500):
    n = random.randint(1, 12)
    root = random_bst(n, lo=-20, hi=20)
    vs = inorder_list(root)
    k = random.randint(1, n)
    t = random.randint(-25, 25) + random.choice([0.1, 0.3, 0.7, 0.9])        # không bao giờ hòa
    assert sorted(answer_closest_k_exercise(root, t, k)) == sorted(closest_k_sort(root, t, k))
    assert answer_closest_k_sorted_output(root, t, k) == sorted(closest_k_sort(root, t, k))
    s = random.randint(-40, 40)
    brute653 = any(vs[i] + vs[j] == s for i in range(n) for j in range(i + 1, n))
    assert answer_two_sum_bst(root, s) == brute653
    arr = sorted(random.choices(range(-10, 11), k=n))                         # có thể có số trùng
    x = random.randint(-12, 12)
    brute658 = sorted(sorted(arr, key=lambda a: (abs(a - x), a))[:k])
    assert answer_find_closest_elements(arr, k, x) == brute658

print("Đáp án Bài 3-6: PASS toàn bộ test (cả 500 bộ ngẫu nhiên)")

Bài 1: [3, 4] [8, 6]
Khởi tạo: pred=[3, 4]  succ=[8, 6]
Bước 1: đỉnh pred=4 (0.4)   đỉnh succ=6 (1.6)   -> lấy  4 (pred)  pred=[3] succ=[8, 6] res=[4]
Bước 2: đỉnh pred=3 (1.4)   đỉnh succ=6 (1.6)   -> lấy  3 (pred)  pred=[1] succ=[8, 6] res=[4, 3]
Bước 3: đỉnh pred=1 (3.4)   đỉnh succ=6 (1.6)   -> lấy  6 (succ)  pred=[1] succ=[8, 7] res=[4, 3, 6]


Đáp án Bài 3-6: PASS toàn bộ test (cả 500 bộ ngẫu nhiên)


## 20. Template cuối cùng nên nhớ

**Hai stack predecessor / successor (cách chính)** — `O(h + k)` time, `O(h)` bộ nhớ thêm:

```python
class Solution:
    def closestKValues(self, root, target, k):
        pred, succ = [], []
        node = root
        while node:                                   # 1) đi đường tìm kiếm, chia hai phía
            if node.val < target:
                pred.append(node); node = node.right
            else:
                succ.append(node); node = node.left

        def next_pred():                              # 2a) lùi: pop, đẩy chuỗi PHẢI của con trái
            node = pred.pop(); cur = node.left
            while cur: pred.append(cur); cur = cur.right
            return node.val

        def next_succ():                              # 2b) tiến: pop, đẩy chuỗi TRÁI của con phải
            node = succ.pop(); cur = node.right
            while cur: succ.append(cur); cur = cur.left
            return node.val

        res = []
        for _ in range(k):                            # 3) trộn từ giữa ra: lấy đỉnh gần hơn
            if not succ or (pred and target - pred[-1].val <= succ[-1].val - target):
                res.append(next_pred())
            else:
                res.append(next_succ())
        return res
```

**Câu thần chú:** *"Đi đường tìm kiếm để chia hai phía. Mỗi lượt lấy đỉnh gần hơn. Lấy bên nào thì đẩy chuỗi bên đó: pred — trái một bước rồi phải hết cỡ; succ — phải một bước rồi trái hết cỡ."*

**Chọn cách theo tình huống:**

- Cây nhị phân **bất kỳ** (không phải BST) → heap kích thước `k`, `O(n log k)`.
- BST, cần lời giải `O(n)` nhanh gọn → in-order + cửa sổ trượt `deque`, dừng sớm.
- BST + nhiều truy vấn trên cùng cây → dựng mảng in-order một lần, mỗi truy vấn `bisect` + hai con trỏ `O(log n + k)`.
- BST cân bằng + follow-up "nhanh hơn `O(n)`" → **hai stack**, `O(h + k) = O(log n + k)`.

## 21. Checklist tự kiểm tra trước khi submit

- Khởi tạo chia node bằng **một** phép so sánh (`node.val < target` → `pred`, còn lại → `succ`) trong **một** lần đi, để giá trị bằng `target` không bị lấy hai lần?
- Sau `pred.pop()`: đi `node.left` rồi **`right` hết cỡ**? Sau `succ.pop()`: đi `node.right` rồi **`left` hết cỡ**?
- Đã kiểm tra stack **rỗng** trước khi đọc `pred[-1]` / `succ[-1]` chưa (`if not succ or (pred and ...)`)?
- So sánh khoảng cách đúng chiều: `target - pred[-1].val` và `succ[-1].val - target` (đều `>= 0`), hoặc dùng `abs`?
- Lời giải **không đệ quy sâu** (cây lệch `10^4` tầng hợp lệ theo constraints)?
- Lặp **đúng `k` lần** và trả về list `k` phần tử (đề đảm bảo `k <= n` nên không bao giờ cả hai stack cùng rỗng giữa chừng)?
- Đã test: Example 1 & 2, `k = n`, `target` nhỏ hơn / lớn hơn mọi giá trị (một stack rỗng từ đầu), `target` trùng giá trị node, cây lệch?
- Đề cho **thứ tự bất kỳ** → khi tự test, so sánh **sau khi sort**; khi sinh test ngẫu nhiên, tránh **hòa ở ranh giới** `k / k+1`.
- Trả lời được follow-up: `O(h + k)`; BST cân bằng → `O(log n + k)`; vì sao không thể tốt hơn `O(k)`.